<a href="https://colab.research.google.com/github/kilianodonell-cmd/Durban/blob/main/Durban_MCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:

# ── Install packages ─────────────────────────────────────────
!pip install geopandas matplotlib contextily fiona pyproj rasterio scipy matplotlib-scalebar -q
!pip install -q geopandas matplotlib contextily fiona pyproj


# ── Standard library ─────────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ── Data processing ──────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Geospatial ───────────────────────────────────────────────
import geopandas as gpd
import rasterio
import rasterio.features
import rasterio.mask
from rasterio.mask import mask as rasterio_mask
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.plot import plotting_extent
from scipy.ndimage import distance_transform_edt

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib_scalebar.scalebar import ScaleBar




print("Setup complete.")

Setup complete.


In [ ]:

# Mount Google Drive (only in Colab)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Setup complete.")

In [ ]:
# ============================================================
# CELL 2 — Config
# ============================================================
# THIS IS THE ONLY CELL YOU NEED TO EDIT.
#
# To add a new factor:
#   1. Upload data to Google Drive in the correct group folder
#   2. Add one new_factor() entry to FACTORS
#   3. Add pairwise comparisons to AHP_COMPARISONS
#   4. Rerun Cells 5, 9, 10, 12, 13
#
# To change scoring bands : edit 'scoring' in the factor entry
# To change AHP weights   : edit AHP_COMPARISONS
# To change scenarios     : edit SCENARIOS
# To change river buffer  : edit RIVER_BUFFER_M
#
# No commenting/uncommenting needed — missing files are
# detected and skipped automatically.
# ============================================================


# ── Root paths ───────────────────────────────────────────────
DATA_ROOT   = '/content/drive/MyDrive/Durban/data'
OUTPUT_ROOT = '/content/drive/MyDrive/Durban/outputs'

# ── Reference paths ──────────────────────────────────────────
CATCHMENTS_PATH = os.path.join(DATA_ROOT, 'reference', 'Major_Catchments_-1661937205605465479(1).gpkg')
BOUNDARY_PATH   = os.path.join(DATA_ROOT, 'reference', 'eThekwini_Municipal_Boundary_-6866066624922810567.gpkg')
UDL_PATH        = os.path.join(DATA_ROOT, 'reference', 'Urban_Development_Line_6297519486847529771.gpkg')
BUILDING_FOOTPRINTS_PATH = os.path.join(DATA_ROOT, 'reference', 'mzinyati_buildings.gpkg')
# ── AOI ──────────────────────────────────────────────────────
AOI_FIELD = 'CAT_NAME'
AOI_VALUE = 'Mzinyati Stream'

# ── Settings ─────────────────────────────────────────────────
TARGET_CRS     = 'EPSG:32736'
CELL_SIZE_M    = 30
RIVER_BUFFER_M = 30

# ── Data folders ─────────────────────────────────────────────
GRP_A = os.path.join(DATA_ROOT, 'group_a_hazards')
GRP_B = os.path.join(DATA_ROOT, 'group_b_infrastructure')
GRP_C = os.path.join(DATA_ROOT, 'group_c_socioeconomic')
GRP_D = os.path.join(DATA_ROOT, 'group_d_community')
GRP_E = os.path.join(DATA_ROOT, 'group_d_environmental')

# ============================================================
# FACTOR HELPER
# ============================================================
def new_factor(group, label, path, ftype, clipped, scored,
               scoring, buffer=None, ffilter=None):
    """
    Create a factor definition dictionary.
    group   : AHP group name
    label   : human readable name for maps
    path    : source file path, list of paths to merge (vectors only),
              or None (for factors clipped separately e.g. health)
    ftype   : 'raster' — reproject, use values directly (already scored 1-5)
               'slope'  — derive slope from DEM
               'vector' — Euclidean distance then reclassify
    clipped : output filename in outputs/clipped/
    scored  : output filename in outputs/rasters/
    scoring : [(max_value, score), ...] or None if already scored 1-5
    buffer  : metres to buffer AOI before clipping (None = AOI boundary)
    ffilter : lambda to filter vector features (None = use all)

    Note: merging multiple paths (list) is only supported for
    vector factors. Rasters must always be single files.
    """
    if isinstance(path, list) and ftype != 'vector':
        raise ValueError(
            f"Factor '{label}': merging (list of paths) is only "
            f"supported for vector factors, not '{ftype}'. "
            f"Keep rasters as separate factors in FACTORS."
        )
    return {
        'group'  : group,
        'label'  : label,
        'path'   : path,
        'type'   : ftype,
        'buffer' : buffer,
        'filter' : ffilter,
        'clipped': clipped,
        'scored' : scored,
        'scoring': scoring,
    }

# ============================================================
# FACTORS REGISTRY
# ============================================================
# All factors defined here regardless of data availability.
# Missing files are detected and skipped automatically.
# To add a factor: add one new_factor() entry below.
# ============================================================

FACTORS = {

    # ── Group A — Hazards ────────────────────────────────────

    'Landslide': new_factor(
        group   = 'Hazards',
        label   = 'Landslide Susceptibility',
        path    = os.path.join(GRP_A, 'landslide_susceptibility.tif'),
        ftype   = 'raster',
        clipped = 'landslide.tif',
        scored  = 'scored_landslide.tif',
        scoring = None,  # already scored 1-5
    ),

    'Slope': new_factor(
        group   = 'Hazards',
        label   = 'Slope (0-30%)',
        path    = os.path.join(GRP_A, 'S30E030_FABDEM_V1-2.tif'),
        ftype   = 'slope',
        clipped = 'dem.tif',
        scored  = 'scored_slope.tif',
        scoring = [(5,1),(10,2),(15,3),(20,4),(30,5)],
    ),

    'Flood': new_factor(
        group   = 'Hazards',
        label   = 'Flood Probability',
        path    = os.path.join(GRP_A, 'flood_extents.tif'),
        ftype   = 'raster',
        clipped = 'flood.tif',
        scored  = 'scored_flood.tif',
        scoring = None,  # already scored 1-5
    ),

    'Community': new_factor(
        group   = 'Hazards',
        label   = 'Community Hazard Map',
        path    = os.path.join(GRP_D, 'community_hazard_map.tif'),
        ftype   = 'raster',
        clipped = 'community.tif',
        scored  = 'scored_community.tif',
        scoring = None,  # already scored 1-5
    ),

    # ── Group B — Infrastructure ─────────────────────────────

    'Primary roads': new_factor(
        group   = 'Infrastructure',
        label   = 'Primary Roads',
        path    = os.path.join(GRP_B, 'Roads_-6417481537698646045(1).gpkg'),
        ftype   = 'vector',
        buffer  = 1000,
        ffilter = lambda gdf: gdf[
            gdf['ROAD_TYPE'].isin(['HWY','RD','AVE','DR']) |
            gdf['ROAD_NAME'].str.match(r'^(A|P|MR)\d', na=False)
        ],
        clipped = 'roads_primary.gpkg',
        scored  = 'scored_roads_primary.tif',
        scoring = [(100,1),(300,2),(600,3),(1000,4),(9999,5)],
    ),

    'Secondary roads': new_factor(
        group   = 'Infrastructure',
        label   = 'Secondary Roads',
        path    = os.path.join(GRP_B, 'Roads_-6417481537698646045(1).gpkg'),
        ftype   = 'vector',
        buffer  = 1000,
        ffilter = lambda gdf: gdf[
            gdf['ROAD_TYPE'].isin(['STR','WAY','CRES','CRL','CRT','PL']) |
            gdf['ROAD_NAME'].str.match(r'^D\d', na=False)
        ],
        clipped = 'roads_secondary.gpkg',
        scored  = 'scored_roads_secondary.tif',
        scoring = [(50,1),(200,2),(500,3),(1000,4),(9999,5)],
    ),

    'Tracks': new_factor(
        group   = 'Infrastructure',
        label   = 'Tracks',
        path    = os.path.join(GRP_B, 'Roads_-6417481537698646045(1).gpkg'),
        ftype   = 'vector',
        buffer  = 500,
        ffilter = lambda gdf: gdf[
            gdf['ROAD_TYPE'].isin(['TRK','WALK','CLS']) |
            gdf['ROAD_TYPE'].isna() |
            (gdf['ROAD_TYPE'] == '')
        ],
        clipped = 'roads_tracks.gpkg',
        scored  = 'scored_roads_tracks.tif',
        scoring = [(50,1),(150,2),(300,3),(500,4),(9999,5)],
    ),

  'Health': new_factor(
    group   = 'Infrastructure',
    label   = 'Health Facilities',
    path    = [
        os.path.join(GRP_B, 'Fixed_Clinics_9054252116514405220.gpkg'),
        os.path.join(GRP_B, 'Mobile_Clinics_5950932655647526625.gpkg'),
        os.path.join(GRP_B, 'Hospitals_1805711158752697858.gpkg'),
    ],
    ftype   = 'vector',
    buffer  = 8000,
    clipped = 'health_facilities.gpkg',
    scored  = 'scored_health.tif',
    scoring = [(500,1),(2000,2),(4000,3),(8000,4),(9999,5)],
),

    'Fire': new_factor(
        group   = 'Infrastructure',
        label   = 'Fire Stations',
        path    = os.path.join(GRP_B, 'Fire_Stations_-6583759099495126262.gpkg'),
        ftype   = 'vector',
        buffer  = 8000,
        clipped = 'fire.gpkg',
        scored  = 'scored_fire.tif',
        scoring = [(500,1),(2000,2),(4000,3),(8000,4),(9999,5)],
    ),

    'Water': new_factor(
        group   = 'Infrastructure',
        label   = 'Water Network',
        path    = os.path.join(GRP_B, 'water_network.gpkg'),
        ftype   = 'vector',
        buffer  = 1000,
        clipped = 'water.gpkg',
        scored  = 'scored_water.tif',
        scoring = [(100,1),(300,2),(600,3),(1000,4),(9999,5)],
    ),

    'Electricity': new_factor(
        group   = 'Infrastructure',
        label   = 'Electricity Network',
        path    = os.path.join(GRP_B, 'electricity_network.gpkg'),
        ftype   = 'vector',
        buffer  = 1000,
        clipped = 'electricity.gpkg',
        scored  = 'scored_electricity.tif',
        scoring = [(100,1),(300,2),(600,3),(1000,4),(9999,5)],
    ),

    # ── Group C — Socioeconomic ──────────────────────────────

    'Settlements': new_factor(
        group   = 'Socioeconomic',
        label   = 'Proximity to Settlements',
        path    = os.path.join(GRP_C, 'settlements.gpkg'),
        ftype   = 'vector',
        buffer  = 3000,
        clipped = 'settlements.gpkg',
        scored  = 'scored_settlements.tif',
        scoring = [(100,1),(500,2),(1000,3),(3000,4),(9999,5)],
    ),

    'Investment nodes': new_factor(
        group   = 'Socioeconomic',
        label   = 'Investment Nodes',
        path    = os.path.join(GRP_C, 'investment_nodes.gpkg'),
        ftype   = 'vector',
        clipped = 'investment_nodes.gpkg',
        scored  = 'scored_investment.tif',
        scoring = [(300,1),(750,2),(1500,3),(3000,4),(9999,5)],
    ),

    'Transport corridors': new_factor(
        group   = 'Socioeconomic',
        label   = 'Transport Corridors',
        path    = os.path.join(GRP_C, 'transport_corridors.gpkg'),
        ftype   = 'vector',
        clipped = 'transport_corridors.gpkg',
        scored  = 'scored_transport.tif',
        scoring = [(100,1),(400,2),(800,3),(1500,4),(9999,5)],
    ),

}

# ============================================================
# CONSTRAINTS
# ============================================================
# Binary exclusion layers — not scored, not weighted.
# Missing files skipped automatically.
# ============================================================
SLOPE_CONSTRAINT_PCT = 30  # cells above this % slope are excluded
CONSTRAINTS = {

    'slope': {
        'label'    : f'Slope > {SLOPE_CONSTRAINT_PCT}%',
        'path'     : None,
        'buffer'   : None,
        'clipped'  : 'dem.tif',
        'type'     : 'slope',
        'threshold': SLOPE_CONSTRAINT_PCT,
    },

    'dmoss': {
        'label'  : "D'MOSS 2018",
        'path'   : os.path.join(GRP_E, 'DMOSS_2018_-896864214986054099.gpkg'),
        'buffer' : None,
        'clipped': 'dmoss.gpkg',
        'type'   : 'vector',
    },

    'river': {
        'label'  : 'Rivers',
        'path'   : os.path.join(GRP_E, 'Rivers_-9177218102417537696.gpkg'),
        'buffer' : RIVER_BUFFER_M,
        'clipped': 'river.gpkg',
        'type'   : 'vector',
    },

    'wetlands': {
        'label'  : 'Wetlands',
        'path'   : os.path.join(GRP_E, 'wetlands.gpkg'),
        'buffer' : None,
        'clipped': 'wetlands.gpkg',
        'type'   : 'vector',
    },

}




# ── Health source layers (merged into Health in Cell 5) ──────
HEALTH_SOURCES = {
    'Fixed Clinics' : os.path.join(GRP_B, 'Fixed_Clinics_9054252116514405220.gpkg'),
    'Mobile Clinics': os.path.join(GRP_B, 'Mobile_Clinics_5950932655647526625.gpkg'),
    'Hospitals'     : os.path.join(GRP_B, 'Hospitals_1805711158752697858.gpkg'),
}

# ============================================================
# AHP PAIRWISE COMPARISONS
# ============================================================
# Format: (factor_i, factor_j, score)
# Saaty scale: 1=equal, 3=moderate, 5=strong, 7=very strong, 9=extreme
# When adding a factor: add comparisons vs all existing
# factors in the same group.
# ============================================================

AHP_COMPARISONS = {

    'group': [
        ('Hazards',        'Infrastructure', 7),
        ('Hazards',        'Socioeconomic',  7),
        ('Infrastructure', 'Socioeconomic',  3),
    ],

    'Hazards': [
        ('Flood',     'Landslide', 1),
        ('Flood',     'Slope',     9),
        ('Landslide', 'Slope',     9),
        ('Community', 'Flood',     7),
        ('Community', 'Landslide', 7),
        ('Community', 'Slope',     7),
    ],

    'Infrastructure': [
        ('Primary roads',   'Secondary roads', 7),
        ('Primary roads',   'Tracks',          9),
        ('Secondary roads', 'Tracks',          7),
        ('Health',          'Primary roads',   9),
        ('Fire',            'Primary roads',   9),
        ('Health',          'Secondary roads', 9),
        ('Fire',            'Secondary roads', 9),
        ('Health',          'Tracks',          9),
        ('Fire',            'Tracks',          9),
        ('Primary roads',   'Water',           3),
        ('Primary roads',   'Electricity',     3),
        ('Water',           'Electricity',     1),
        ('Health',          'Water',           9),
        ('Fire',            'Water',           9),
        ('Health',          'Electricity',     9),
        ('Fire',            'Electricity',     9),
        ('Health',          'Fire',            3),
    ],

    'Socioeconomic': [
        ('Investment nodes', 'Transport corridors', 7),
        ('Settlements',      'Investment nodes',    7),
        ('Settlements',      'Transport corridors', 7),
    ],

}

# ============================================================
# SCENARIOS
# ============================================================
# Each scenario defines group-level weights.
# Within-group weights always come from FAHP (Cell 9).
# Weights must sum to 1.0.
# Add new scenarios by copying an existing block.
# ============================================================

SCENARIOS = {

    'hazard_focused': {
        'description'    : 'Hazard-focused — safety priority',
        'Hazards'        : 0.76,
        'Infrastructure' : 0.16,
        'Socioeconomic'  : 0.08,
    },

    'balanced': {
        'description'    : 'Balanced — equal group weights',
        'Hazards'        : 0.33,
        'Infrastructure' : 0.33,
        'Socioeconomic'  : 0.34,
    },

    'infrastructure_focused': {
        'description'    : 'Infrastructure-focused — service accessibility priority',
        'Hazards'        : 0.20,
        'Infrastructure' : 0.60,
        'Socioeconomic'  : 0.20,
    },

}

# ============================================================
# DERIVED LOOKUPS — auto-generated from FACTORS
# Do not edit these
# ============================================================

LAYER_NAMES = {f: v['label'] for f, v in FACTORS.items()}
LAYER_NAMES.update({k: v['label'] for k, v in CONSTRAINTS.items()})
LAYER_NAMES.update({
    'catchments': 'Major Catchments',
    'boundary'  : 'eThekwini Municipal Boundary',
    'udl'       : 'Urban Development Line',
    'aoi'       : 'Mzinyati Stream Catchment',
    'dem'       : 'DEM (FABDEM)',
})

GROUP_MEMBERSHIP = {}
for factor, props in FACTORS.items():
    group = props['group']
    if group not in GROUP_MEMBERSHIP:
        GROUP_MEMBERSHIP[group] = []
    GROUP_MEMBERSHIP[group].append(factor)

# ============================================================
# OUTPUTS
# ============================================================

os.makedirs(os.path.join(OUTPUT_ROOT, 'clipped'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT, 'rasters'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT, 'maps'),    exist_ok=True)

# ── Derived path constants (use these everywhere — never rebuild inline) ──────
CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')
RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
MAPS_DIR    = os.path.join(OUTPUT_ROOT, 'maps')

# ── Validation ───────────────────────────────────────────────
for name, scenario in SCENARIOS.items():
    weights = {k: v for k, v in scenario.items() if k != 'description'}
    total   = sum(weights.values())
    assert abs(total - 1.0) < 0.01, \
        f"Weights in '{name}' do not sum to 1.0 (got {total:.3f})"

# ── Summary ──────────────────────────────────────────────────
print("Config loaded.")
print(f"  Data root:    {DATA_ROOT}")
print(f"  Output root:  {OUTPUT_ROOT}")
print(f"  CRS:          {TARGET_CRS}")
print(f"  Cell size:    {CELL_SIZE_M}m")
print(f"  River buffer: {RIVER_BUFFER_M}m")
print(f"\n  Factors defined: {len(FACTORS)}")
for group, members in GROUP_MEMBERSHIP.items():
    print(f"    {group:20} {', '.join(members)}")
print(f"\n  Constraints defined: {len(CONSTRAINTS)}")
print(f"  Scenarios defined:   {len(SCENARIOS)}")




# ── Export config for Field Map Tool ─────────────────────────

field_map_config = {
    'OUTPUT_ROOT' : OUTPUT_ROOT,
    'TARGET_CRS'  : TARGET_CRS,
    'SCENARIOS'   : {k: v['description'] for k, v in SCENARIOS.items()},
    'SUIT_COLORS' : ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'],
    'SUIT_LABELS' : [l.split(' — ')[1] for l in [
        '1 — Highly suitable', '2 — Suitable', '3 — Moderate',
        '4 — Unsuitable', '5 — Highly unsuitable'
    ]],
    'AOI_FIELD'   : AOI_FIELD,
    'AOI_VALUE'   : AOI_VALUE,
    'CATCHMENTS_PATH': CATCHMENTS_PATH,
    'BOUNDARY_PATH'  : BOUNDARY_PATH,
}

config_path = os.path.join(OUTPUT_ROOT, 'field_map_config.json')
with open(config_path, 'w') as f:
    json.dump(field_map_config, f, indent=2)

print(f"  ✓ Field map config saved → field_map_config.json")

In [18]:
# ============================================================
# CELL 2b — Visual Style and Plotting Functions
# ============================================================
# ALL visual settings and plotting functions defined here.
# No style or layout code anywhere else in the notebook.
#
# To change any visual aspect:
#   edit here only — all cells update automatically.
#
# Available functions:
#   plot_map(data, title, filename)
#   plot_constraint_map(mask, filename)
#   plot_difference_map(diff, title, filename)
#   plot_comparison_map(scenario_maps, scenario_names)
#   plot_distribution_chart(data, label, filename_prefix)
#   plot_scenario_comparison_chart(scenario_maps)
#   print_distribution_table(data, label)
# ============================================================

# ── Suitability colour scheme ─────────────────────────────────
SUIT_COLORS = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
SUIT_LABELS = [
    '1 — Highly suitable',
    '2 — Suitable',
    '3 — Moderate',
    '4 — Unsuitable',
    '5 — Highly unsuitable',
]
SUIT_SCORES = [1, 2, 3, 4, 5]

# ── Scenario comparison colours ───────────────────────────────
SCENARIO_COLORS = ['#2166ac', '#f4a582', '#1a9641']

# ── Colourbar ─────────────────────────────────────────────────
CBAR_LABEL      = '1 = Highly Suitable    5 = Unsuitable'
CBAR_TICKLABELS = [
    '1 Highly suitable', '2 Suitable', '3 Moderate',
    '4 Unsuitable',      '5 Highly unsuitable',
]
CBAR_SHRINK = 0.8

# ── Map elements ──────────────────────────────────────────────
SHOW_NORTH_ARROW    = True
SHOW_SCALE_BAR      = True
SHOW_GRID           = True
AOI_BOUNDARY_COLOR  = 'black'
AOI_BOUNDARY_WIDTH  = 1.5
AOI_BUFFER_FRACTION = 0.05

# ── Figure sizes ──────────────────────────────────────────────
MAP_FIGSIZE   = (8, 9)
CHART_FIGSIZE = (9, 5)

# ── Font sizes ────────────────────────────────────────────────
FONT_TITLE      = 11
FONT_AXIS_LABEL = 9
FONT_TICK       = 7
FONT_LEGEND     = 8
FONT_CBAR_LABEL = 9
FONT_CBAR_TICK  = 8

# ── Output resolution ─────────────────────────────────────────
DPI = 300

# ── Grid style ────────────────────────────────────────────────
GRID_ALPHA     = 0.2
GRID_LINEWIDTH = 0.5

# ── Chart style ───────────────────────────────────────────────
CHART_BAR_WIDTH = 0.6
CHART_BAR_ALPHA = 0.9

# ── Difference map ────────────────────────────────────────────
DIFF_CMAP  = 'RdBu_r'
DIFF_VMIN  = -2
DIFF_VMAX  = 2
DIFF_LABEL = 'Score difference (positive = scenario A more suitable)'

# ============================================================
# INTERNAL HELPERS
# ============================================================

try:
    from matplotlib_scalebar.scalebar import ScaleBar
    HAS_SCALEBAR = True
except ImportError:
    HAS_SCALEBAR = False

def _get_extent(filename):
    """Get plotting extent from saved raster file."""
    with rasterio.open(os.path.join(OUTPUT_ROOT, 'rasters', filename)) as src:
        return plotting_extent(src)

def _get_aoi_bounds():
    """Get AOI bounds with buffer for map zoom."""
    minx, miny, maxx, maxy = aoi.total_bounds
    buf = (maxx - minx) * AOI_BUFFER_FRACTION
    return minx - buf, maxx + buf, miny - buf, maxy + buf

def _add_map_furniture(ax):
    """Add north arrow, scale bar and grid to map axis."""
    if SHOW_NORTH_ARROW:
        ax.annotate(
            'N', xy=(0.96, 0.96), xytext=(0.96, 0.88),
            xycoords='axes fraction',
            fontsize=12, fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='black', lw=2),
            annotation_clip=False
        )
    if SHOW_SCALE_BAR and HAS_SCALEBAR:
        ax.add_artist(ScaleBar(
            1, 'm', length_fraction=0.25,
            location='lower left', frameon=True,
            box_alpha=0.8, font_properties={'size': FONT_LEGEND}
        ))
    if SHOW_GRID:
        ax.grid(True, alpha=GRID_ALPHA, linewidth=GRID_LINEWIDTH)

def _add_colourbar(fig, ax, im):
    """Add standard suitability colourbar."""
    cbar = fig.colorbar(im, ax=ax, shrink=CBAR_SHRINK)
    cbar.set_label(CBAR_LABEL, fontsize=FONT_CBAR_LABEL)
    cbar.set_ticks(SUIT_SCORES)
    cbar.set_ticklabels(CBAR_TICKLABELS)
    cbar.ax.tick_params(labelsize=FONT_CBAR_TICK)
    return cbar

def _style_map_ax(ax, x_lim, x_lim2, y_lim, y_lim2):
    """Apply standard map axis styling."""
    ax.set_xlim(x_lim, x_lim2)
    ax.set_ylim(y_lim, y_lim2)
    ax.set_xlabel('Easting (m)', fontsize=FONT_AXIS_LABEL)
    ax.set_ylabel('Northing (m)', fontsize=FONT_AXIS_LABEL)
    ax.tick_params(labelsize=FONT_TICK)
    ax.set_facecolor('white')

def _draw_suitability(ax, data, extent):
    """Draw suitability raster with constraint overlay. Returns im."""
    cmap = ListedColormap(SUIT_COLORS)
    norm = BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], cmap.N)

    # Constrained areas white
    constraint_plot = np.where(
        (aoi_raster_mask == 1) & (constraint_mask == 0),
        1.0, np.nan
    )
    ax.imshow(
        constraint_plot,
        cmap='binary_r', vmin=0, vmax=1,
        extent=extent, origin='upper', alpha=1.0
    )

    # Suitability data
    plot_data = data.copy()
    plot_data[aoi_raster_mask == 0] = np.nan
    im = ax.imshow(
        plot_data, cmap=cmap, norm=norm,
        extent=extent, origin='upper'
    )

    # AOI boundary
    aoi.to_crs(TARGET_CRS).boundary.plot(
        ax=ax, color=AOI_BOUNDARY_COLOR, linewidth=AOI_BOUNDARY_WIDTH
    )
    return im

def _save_fig(fig, filename):
    """Save figure to outputs/maps/"""
    out_path = os.path.join(OUTPUT_ROOT, 'maps', filename)
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight')
    print(f"  ✓ Saved → {filename}")

def _distribution_stats(data):
    """Calculate distribution statistics from suitability data."""
    total_aoi   = int(aoi_raster_mask.sum())
    # Buildable = inside AOI AND not constrained
    buildable_mask = (aoi_raster_mask == 1) & (constraint_mask == 1)
    buildable   = buildable_mask & ~np.isnan(data)
    total_build = int(buildable_mask.sum())
    total_excl  = total_aoi - total_build
    counts, pct_b, pct_a = [], [], []
    for score in SUIT_SCORES:
        count = int(((data >= score-0.5) & (data < score+0.5) & buildable).sum())
        counts.append(count)
        pct_b.append(count / total_build * 100 if total_build > 0 else 0)
        pct_a.append(count / total_aoi   * 100 if total_aoi   > 0 else 0)
    return total_aoi, total_build, total_excl, counts, pct_b, pct_a
# ============================================================
# PUBLIC PLOTTING FUNCTIONS
# Called by Cells 11, 12, 13
# ============================================================

def plot_map(data, title, filename):
    """
    Plot suitability map.
    data     : numpy array of suitability scores
    title    : map title string
    filename : output filename (saved to outputs/maps/)
    """
    extent              = _get_extent(filename.replace('.png', '.tif'))
    x0, x1, y0, y1     = _get_aoi_bounds()

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    fig.patch.set_facecolor('white')

    im = _draw_suitability(ax, data, extent)
    _style_map_ax(ax, x0, x1, y0, y1)
    _add_map_furniture(ax)
    _add_colourbar(fig, ax, im)

    ax.set_title(title, fontsize=FONT_TITLE, pad=10)
    plt.tight_layout()
    _save_fig(fig, filename)
    plt.show()

def plot_constraint_map(mask_data, filename):
    """
    Plot constraint mask map.
    Green = buildable (SUIT_COLORS[0])
    Red   = excluded  (SUIT_COLORS[4])
    """
    extent          = _get_extent('constraint_mask.tif')
    x0, x1, y0, y1 = _get_aoi_bounds()

    total_cells     = int(aoi_raster_mask.sum())
    buildable_cells = int(((mask_data == 1) & (aoi_raster_mask == 1)).sum())
    excluded_cells  = total_cells - buildable_cells

    display = np.full(mask_data.shape, np.nan, dtype='float32')
    display[(aoi_raster_mask == 1) & (mask_data == 1)] = 1
    display[(aoi_raster_mask == 1) & (mask_data == 0)] = 5

    cmap_mask = ListedColormap([SUIT_COLORS[0], SUIT_COLORS[4]])
    norm_mask = BoundaryNorm([0.5, 3.0, 5.5], cmap_mask.N)

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    ax.imshow(display, cmap=cmap_mask, norm=norm_mask, extent=extent, origin='upper')
    aoi.to_crs(TARGET_CRS).boundary.plot(
        ax=ax, color=AOI_BOUNDARY_COLOR, linewidth=AOI_BOUNDARY_WIDTH
    )
    _style_map_ax(ax, x0, x1, y0, y1)
    _add_map_furniture(ax)

    legend_elements = [
        mpatches.Patch(
            facecolor=SUIT_COLORS[0], edgecolor='grey', linewidth=0.5,
            label=f'Buildable ({buildable_cells/total_cells*100:.1f}%)'
        ),
        mpatches.Patch(
            facecolor=SUIT_COLORS[4], edgecolor='grey', linewidth=0.5,
            label=f'Excluded ({excluded_cells/total_cells*100:.1f}%)'
        ),
    ]
    ax.legend(handles=legend_elements, loc='lower right',
              fontsize=FONT_LEGEND, framealpha=0.9)

    ax.set_title(f'Constraint Mask\n{LAYER_NAMES["aoi"]}',
                 fontsize=FONT_TITLE, pad=10)
    plt.tight_layout()
    _save_fig(fig, filename)
    plt.show()

def plot_difference_map(diff, title, filename):
    """
    Plot difference map between two scenarios.
    diff     : numpy array (map_a - map_b)
    title    : map title string
    filename : output filename
    """
    extent          = _get_extent(filename.replace('.png', '.tif')
                                  .replace('difference_', 'suitability_')
                                  .split('_vs_')[0] + '.tif')
    x0, x1, y0, y1 = _get_aoi_bounds()

    plot_diff = diff.copy()
    plot_diff[aoi_raster_mask == 0] = np.nan

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    im = ax.imshow(
        plot_diff, cmap=DIFF_CMAP, vmin=DIFF_VMIN, vmax=DIFF_VMAX,
        extent=extent, origin='upper'
    )
    aoi.to_crs(TARGET_CRS).boundary.plot(
        ax=ax, color=AOI_BOUNDARY_COLOR, linewidth=AOI_BOUNDARY_WIDTH
    )
    _style_map_ax(ax, x0, x1, y0, y1)
    _add_map_furniture(ax)

    cbar = fig.colorbar(im, ax=ax, shrink=CBAR_SHRINK)
    cbar.set_label(DIFF_LABEL, fontsize=FONT_CBAR_LABEL)
    cbar.ax.tick_params(labelsize=FONT_CBAR_TICK)

    ax.set_title(title, fontsize=FONT_TITLE, pad=10)
    plt.tight_layout()
    _save_fig(fig, filename)
    plt.show()

def plot_comparison_map(scenario_maps, scenario_names):
    """
    Side by side comparison map for all scenarios.
    scenario_maps   : dict {scenario_name: numpy array}
    scenario_names  : list of scenario names in order
    """
    n_sc  = len(scenario_maps)
    fig, axes = plt.subplots(
        1, n_sc,
        figsize=(MAP_FIGSIZE[0] * n_sc, MAP_FIGSIZE[1])
    )
    if n_sc == 1:
        axes = [axes]
    fig.patch.set_facecolor('white')

    x0, x1, y0, y1 = _get_aoi_bounds()
    im_last = None

    for ax, scenario_name in zip(axes, scenario_names):
        description = SCENARIOS[scenario_name]['description']
        data        = scenario_maps[scenario_name]
        filename    = f"suitability_{scenario_name}.tif"
        extent      = _get_extent(filename)

        im = _draw_suitability(ax, data, extent)
        _style_map_ax(ax, x0, x1, y0, y1)
        _add_map_furniture(ax)
        ax.set_title(description, fontsize=FONT_TITLE - 1, pad=8)
        im_last = im

    # Shared colourbar
      # Shared colourbar

    fig.suptitle(
        f'Housing Suitability — Scenario Comparison\n{LAYER_NAMES["aoi"]}',
        fontsize=FONT_TITLE, y=1.01
    )
    plt.tight_layout()
    _save_fig(fig, 'scenario_comparison.png')
    plt.show()

def plot_distribution_chart(data, label, filename_prefix):
    """
    Vertical bar chart — % of buildable area per suitability class.
    Labels on bars show % of total AOI.
    """
    total_aoi, total_build, total_excl, counts, pct_b, pct_a = \
        _distribution_stats(data)

    fig, ax = plt.subplots(figsize=CHART_FIGSIZE)
    fig.patch.set_facecolor('white')

    x_labels = [l.split(' — ')[1] for l in SUIT_LABELS]
    bars = ax.bar(
        x_labels, pct_b,
        color=SUIT_COLORS, edgecolor='white',
        linewidth=0.5, width=CHART_BAR_WIDTH, alpha=CHART_BAR_ALPHA
    )

    for bar, pb, pa in zip(bars, pct_b, pct_a):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{pb:.1f}%\n({pa:.1f}% AOI)",
            ha='center', va='bottom',
            fontsize=FONT_LEGEND, color='#333333'
        )

    ax.set_ylabel('% of buildable area', fontsize=FONT_AXIS_LABEL)
    ax.set_ylim(0, max(pct_b) * 1.35 if max(pct_b) > 0 else 100)
    ax.set_title(
        f'Suitability Class Distribution — {label}\n'
        f'Columns: % of buildable area  |  Labels: % of total AOI',
        fontsize=FONT_TITLE - 1
    )
    ax.tick_params(labelsize=FONT_TICK + 1)
    ax.grid(True, axis='y', alpha=GRID_ALPHA, linewidth=GRID_LINEWIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    _save_fig(fig, f"distribution_{filename_prefix}.png")
    plt.show()

def plot_scenario_comparison_chart(scenario_maps):
    """
    Grouped vertical bar chart comparing suitability distribution
    across all scenarios. One colour per scenario from SCENARIO_COLORS.
    """
    scenario_names = list(scenario_maps.keys())
    descriptions   = [SCENARIOS[n]['description'].split(' — ')[0]
                      for n in scenario_names]
    n_sc           = len(scenario_names)
    x              = np.arange(len(SUIT_SCORES))
    bar_width      = CHART_BAR_WIDTH / n_sc

    fig, ax = plt.subplots(figsize=CHART_FIGSIZE)
    fig.patch.set_facecolor('white')

    for idx, (scenario_name, final_map) in enumerate(scenario_maps.items()):
        buildable = ~np.isnan(final_map)
        total_b   = buildable.sum()
        pcts      = [
            int(((final_map >= s-0.5) & (final_map < s+0.5) & buildable).sum())
            / total_b * 100 if total_b > 0 else 0
            for s in SUIT_SCORES
        ]
        ax.bar(
            x + idx * bar_width - (n_sc - 1) * bar_width / 2,
            pcts,
            width     = bar_width,
            label     = descriptions[idx],
            color     = SCENARIO_COLORS[idx % len(SCENARIO_COLORS)],
            edgecolor = 'white',
            linewidth = 0.5,
            alpha     = CHART_BAR_ALPHA,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([l.split(' — ')[1] for l in SUIT_LABELS],
                       fontsize=FONT_TICK + 1)
    ax.set_ylabel('% of buildable area', fontsize=FONT_AXIS_LABEL)
    ax.set_title(
        f'Suitability Distribution by Scenario\n{LAYER_NAMES["aoi"]}',
        fontsize=FONT_TITLE
    )
    ax.legend(fontsize=FONT_LEGEND, framealpha=0.9)
    ax.grid(True, axis='y', alpha=GRID_ALPHA, linewidth=GRID_LINEWIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=FONT_TICK)
    plt.tight_layout()

    _save_fig(fig, 'scenario_distribution_comparison.png')
    plt.show()

def print_distribution_table(data, label):
    """
    Print suitability distribution as formatted text table.
    Note: WLC produces continuous scores — classes assigned
    using nearest integer boundaries (±0.5).
    """
    total_aoi, total_build, total_excl, counts, pct_b, pct_a = \
        _distribution_stats(data)

    print(f"\n  Suitability Distribution — {label}")
    print(f"  Note: continuous WLC scores classified using ±0.5 boundaries")
    print(f"  {'─' * 65}")
    print(f"  Total AOI area:   {total_aoi:>8,} cells  (100.0%)")
    print(f"  Buildable area:   {total_build:>8,} cells  ({total_build/total_aoi*100:.1f}% of AOI)")
    print(f"  Excluded area:    {total_excl:>8,} cells  ({total_excl/total_aoi*100:.1f}% of AOI)")
    print(f"  {'─' * 65}")
    print(f"  {'Score':<6} {'Class':<22} {'Cells':>8} {'% Buildable':>12} {'% AOI':>8}")
    print(f"  {'─' * 65}")
    for i, score in enumerate(SUIT_SCORES):
        print(f"  {score:<6} {SUIT_LABELS[i]:<22} {counts[i]:>8,} {pct_b[i]:>11.1f}% {pct_a[i]:>7.1f}%")
    print(f"  {'─' * 65}")



# ── Raster template (set once, reused everywhere) ─────────────
def get_raster_template():
    """Derive canonical transform / width / height from the clipped DEM.
    Call once after Cell 5 (clip) has run; result stored as module-level
    globals so no downstream cell needs to redefine this."""
    with rasterio.open(os.path.join(CLIPPED_DIR, 'dem.tif')) as src:
        if str(src.crs) != TARGET_CRS:
            from rasterio.warp import calculate_default_transform
            _t, _w, _h = calculate_default_transform(
                src.crs, TARGET_CRS, src.width, src.height, *src.bounds
            )
        else:
            _t, _w, _h = src.transform, src.width, src.height
    return _t, _w, _h

# ── Constraint mask helper ────────────────────────────────────
def build_constraint_mask(key, props):
    """Return boolean mask (True = excluded/constrained) for a single
    constraint entry.  Uses globals: transform, width, height,
    CLIPPED_DIR, TARGET_CRS (all set by Cell 11 / Cell 12)."""
    ctype        = props['type']
    clipped_path = os.path.join(CLIPPED_DIR, props['clipped'])

    if ctype == 'slope':
        with rasterio.open(clipped_path) as src:
            dem_data = np.zeros((height, width), dtype='float32')
            reproject(
                source        = rasterio.band(src, 1),
                destination   = dem_data,
                src_transform = src.transform,
                src_crs       = src.crs,
                dst_transform = transform,
                dst_crs       = TARGET_CRS,
                resampling    = Resampling.bilinear
            )
        pixel_size = abs(transform[0])
        dz_dx      = np.gradient(dem_data, pixel_size, axis=1)
        dz_dy      = np.gradient(dem_data, pixel_size, axis=0)
        slope_pct  = np.sqrt(dz_dx**2 + dz_dy**2) * 100
        return slope_pct > props['threshold']

    elif ctype == 'vector':
        gdf = gpd.read_file(clipped_path).to_crs(TARGET_CRS)
        if props['buffer']:
            gdf = gdf.copy()
            gdf['geometry'] = gdf.geometry.buffer(props['buffer'])
        shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
        burned = rasterio.features.rasterize(
            shapes,
            out_shape = (height, width),
            transform = transform,
            fill      = 0,
            dtype     = 'uint8'
        )
        return burned == 1

    return np.zeros((height, width), dtype=bool)

# ── Validation ───────────────────────────────────────────────
assert len(SUIT_COLORS) == len(SUIT_LABELS) == len(SUIT_SCORES), \
    "SUIT_COLORS, SUIT_LABELS and SUIT_SCORES must have the same length"
assert len(CBAR_TICKLABELS) == len(SUIT_SCORES), \
    "CBAR_TICKLABELS must match SUIT_SCORES length"

print("Visual style loaded.")
print(f"  Map size:     {MAP_FIGSIZE}  |  Chart size: {CHART_FIGSIZE}")
print(f"  DPI:          {DPI}")
print(f"  North arrow:  {SHOW_NORTH_ARROW}  |  Scale bar: {SHOW_SCALE_BAR}  |  Grid: {SHOW_GRID}")
print(f"\n  Available functions:")
print(f"    plot_map(data, title, filename)")
print(f"    plot_constraint_map(mask_data, filename)")
print(f"    plot_difference_map(diff, title, filename)")
print(f"    plot_comparison_map(scenario_maps, scenario_names)")
print(f"    plot_distribution_chart(data, label, filename_prefix)")
print(f"    plot_scenario_comparison_chart(scenario_maps)")
print(f"    print_distribution_table(data, label)")

Visual style loaded.
  Map size:     (8, 9)  |  Chart size: (9, 5)
  DPI:          300
  North arrow:  True  |  Scale bar: True  |  Grid: True

  Available functions:
    plot_map(data, title, filename)
    plot_constraint_map(mask_data, filename)
    plot_difference_map(diff, title, filename)
    plot_comparison_map(scenario_maps, scenario_names)
    plot_distribution_chart(data, label, filename_prefix)
    plot_scenario_comparison_chart(scenario_maps)
    print_distribution_table(data, label)


In [ ]:
# ============================================================
# CELL 3 — Data Inventory
# Lists all files found in the data directory.
# ============================================================

for folder, subfolders, files in os.walk(DATA_ROOT):
    rel = os.path.relpath(folder, DATA_ROOT)
    print(f"\n📁 {rel}")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(folder, f)) / (1024 * 1024)
        print(f"   {f}  ({size_mb:.1f} MB)")

In [ ]:
# ============================================================
# CELL 4 — Build AOI and Plot Context Map
# Loads reference layers, filters study area catchment,
# and plots the context map.
# Change AOI_FIELD, AOI_VALUE and LAYER_NAMES in Cell 2
# to use a different study area.
# ============================================================


# ── Load reference layers ────────────────────────────────────
print("Loading reference layers...")

catchments = gpd.read_file(CATCHMENTS_PATH).to_crs(TARGET_CRS)
boundary   = gpd.read_file(BOUNDARY_PATH).to_crs(TARGET_CRS)
udl        = gpd.read_file(UDL_PATH).to_crs(TARGET_CRS)

print(f"  ✓ {LAYER_NAMES['catchments']} — {len(catchments)} features")
print(f"  ✓ {LAYER_NAMES['boundary']} — {len(boundary)} features")
print(f"  ✓ {LAYER_NAMES['udl']} — {len(udl)} features")

# ── Build AOI ────────────────────────────────────────────────
print("\nBuilding AOI...")

aoi_raw = catchments[catchments[AOI_FIELD] == AOI_VALUE]
aoi     = aoi_raw.dissolve().reset_index(drop=True)

print(f"  ✓ {LAYER_NAMES['aoi']}")
print(f"    Features dissolved: {len(aoi_raw)} → 1")
print(f"    Area: {aoi.geometry.area.sum() / 1e6:.2f} km²")
print(f"    CRS: {aoi.crs}")

# ── Plot context map ─────────────────────────────────────────
print("\nPlotting context map...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# ── Left panel: full municipal context ───────────────────────
boundary.plot(
    ax=ax1,
    facecolor='#f9f9f9',
    edgecolor='black',
    linewidth=1.5,
    linestyle='--',
    zorder=1
)
catchments.plot(
    ax=ax1,
    facecolor='#f0f0f0',
    edgecolor='#cccccc',
    linewidth=0.3,
    zorder=2
)
udl.plot(
    ax=ax1,
    facecolor='none',
    edgecolor='#e67e22',
    linewidth=1.5,
    zorder=3
)
aoi.plot(
    ax=ax1,
    facecolor='#2ecc71',
    edgecolor='#27ae60',
    linewidth=2,
    alpha=0.8,
    zorder=4
)

ax1.set_title(
    f'Municipal Context\n{LAYER_NAMES["aoi"]} highlighted',
    fontsize=11
)
ax1.set_xlabel('Easting (m)', fontsize=9)
ax1.set_ylabel('Northing (m)', fontsize=9)
ax1.tick_params(labelsize=7)
ax1.grid(True, alpha=0.3, linewidth=0.5)

legend_elements = [
    mpatches.Patch(facecolor='#f9f9f9', edgecolor='black',   linestyle='--', label=LAYER_NAMES['boundary']),
    mpatches.Patch(facecolor='#f0f0f0', edgecolor='#cccccc',                 label=LAYER_NAMES['catchments']),
    mpatches.Patch(facecolor='none',    edgecolor='#e67e22',                 label=LAYER_NAMES['udl']),
    mpatches.Patch(facecolor='#2ecc71', edgecolor='#27ae60',                 label=LAYER_NAMES['aoi']),
]
ax1.legend(handles=legend_elements, loc='lower right', fontsize=8, framealpha=0.9)

# ── Right panel: zoomed to AOI ───────────────────────────────
catchments.plot(
    ax=ax2,
    facecolor='#f0f0f0',
    edgecolor='#cccccc',
    linewidth=0.3,
    zorder=1
)
udl.plot(
    ax=ax2,
    facecolor='none',
    edgecolor='#e67e22',
    linewidth=1.5,
    zorder=2
)
aoi.plot(
    ax=ax2,
    facecolor='#2ecc71',
    edgecolor='#27ae60',
    linewidth=2,
    alpha=0.8,
    zorder=3
)

# Zoom to AOI extent with small buffer
minx, miny, maxx, maxy = aoi.total_bounds
buffer = (maxx - minx) * 0.1
ax2.set_xlim(minx - buffer, maxx + buffer)
ax2.set_ylim(miny - buffer, maxy + buffer)

ax2.set_title(
    f'Study Area\n{LAYER_NAMES["aoi"]}',
    fontsize=11
)
ax2.set_xlabel('Easting (m)', fontsize=9)
ax2.set_ylabel('Northing (m)', fontsize=9)
ax2.tick_params(labelsize=7)
ax2.grid(True, alpha=0.3, linewidth=0.5)

ax2.legend(handles=legend_elements[1:], loc='lower right', fontsize=8, framealpha=0.9)

# ── Overall title ────────────────────────────────────────────
fig.suptitle('Study Area Context Map', fontsize=14, y=1.01)

plt.tight_layout()

# Save
output_path = os.path.join(OUTPUT_ROOT, 'maps', 'context_map.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n  ✓ Map saved to: {output_path}")

plt.show()

In [ ]:
# ============================================================
# CELL 5 — Clip All Layers to AOI
# Reads FACTORS and CONSTRAINTS from Cell 2.
# Adding a new factor or constraint in Cell 2 is
# automatically clipped here — no changes needed in this cell.
#
# Handles three path types automatically:
#   single path  — clip directly
#   list of paths — merge then clip (vectors only)
#   None         — skip (factor clipped elsewhere)
#
# Skips layers already clipped to save time.
# ============================================================


warnings.filterwarnings('ignore')

CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')
os.makedirs(CLIPPED_DIR, exist_ok=True)

already_clipped = set(os.listdir(CLIPPED_DIR))

def needs_clipping(filename):
    return filename not in already_clipped

# ── Clip raster ──────────────────────────────────────────────
def clip_raster(path, clipped_filename, label):
    if not needs_clipping(clipped_filename):
        print(f"  ↷ {label:35} already clipped, skipped")
        return
    if not os.path.exists(path):
        print(f"  ✗ {label:35} file not found in Drive, skipped")
        return
    with rasterio.open(path) as src:
        aoi_proj = aoi.to_crs(src.crs)
        shapes   = [json.loads(aoi_proj.geometry.to_json())['features'][0]['geometry']]
        out_img, out_transform = rasterio_mask(src, shapes, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            'driver'   : 'GTiff',
            'height'   : out_img.shape[1],
            'width'    : out_img.shape[2],
            'transform': out_transform,
        })
    out = os.path.join(CLIPPED_DIR, clipped_filename)
    with rasterio.open(out, 'w', **out_meta) as dst:
        dst.write(out_img)
    print(f"  ✓ {label:35} clipped raster → {clipped_filename}")

# ── Clip single vector ───────────────────────────────────────
def clip_single_vector(path, label, buffer, ffilter, aoi):
    if not os.path.exists(path):
        print(f"  ✗ {label:35} file not found in Drive, skipped")
        return None
    gdf     = gpd.read_file(path).to_crs(TARGET_CRS)
    if ffilter:
        gdf = ffilter(gdf)
    clip_to = aoi.buffer(buffer).to_frame('geometry').set_crs(TARGET_CRS) if buffer else aoi
    return gpd.clip(gdf, clip_to)

# ── Clip vector (single or merged) ───────────────────────────
def clip_vector(path, clipped_filename, label, buffer=None, ffilter=None):
    if not needs_clipping(clipped_filename):
        print(f"  ↷ {label:35} already clipped, skipped")
        return

    out = os.path.join(CLIPPED_DIR, clipped_filename)

    # ── Single path ──────────────────────────────────────────
    if isinstance(path, str):
        result = clip_single_vector(path, label, buffer, ffilter, aoi)
        if result is None:
            return
        result.to_file(out, driver='GPKG')
        buf_note = f" (buffer: {buffer}m)" if buffer else ""
        print(f"  ✓ {label:35} {len(result)} features → {clipped_filename}{buf_note}")

    # ── List of paths — merge then clip ──────────────────────
    elif isinstance(path, list):
        frames = []
        for p in path:
            if not os.path.exists(p):
                print(f"  ✗ {label:35} source not found: {os.path.basename(p)}, skipped")
                continue
            gdf = gpd.read_file(p).to_crs(TARGET_CRS)
            gdf['source'] = os.path.basename(p)
            frames.append(gdf[['geometry', 'source']])
            print(f"       + {os.path.basename(p):30} {len(gdf)} features")

        if not frames:
            print(f"  ✗ {label:35} no source files found, skipped")
            return

        merged  = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=TARGET_CRS)
        clip_to = aoi.buffer(buffer).to_frame('geometry').set_crs(TARGET_CRS) if buffer else aoi
        clipped = gpd.clip(merged, clip_to)
        clipped.to_file(out, driver='GPKG')
        buf_note = f" (buffer: {buffer}m)" if buffer else ""
        print(f"  ✓ {label:35} {len(clipped)} features merged → {clipped_filename}{buf_note}")

# ============================================================
# CLIP FACTORS
# ============================================================

print("Clipping factors to AOI...")
print(f"  AOI: {LAYER_NAMES['aoi']}")
print(f"  Already clipped: {len(already_clipped)} files found\n")

current_group = None

for factor, props in FACTORS.items():

    # Print group header when group changes
    if props['group'] != current_group:
        current_group = props['group']
        print(f"\n{current_group}")

    label   = props['label']
    ftype   = props['type']
    path    = props['path']
    clipped = props['clipped']
    buffer  = props['buffer']
    ffilter = props['filter']

    # Skip factors with no path — handled separately or not needed
    if path is None:
        print(f"  ↷ {label:35} no path defined, skipped")
        continue

    if ftype in ('raster', 'slope'):
        clip_raster(path, clipped, label)
    elif ftype == 'vector':
        clip_vector(path, clipped, label, buffer, ffilter)

# ============================================================
# CLIP CONSTRAINTS
# ============================================================

print(f"\nConstraints")

for key, props in CONSTRAINTS.items():
    label   = props['label']
    path    = props['path']
    clipped = props['clipped']
    buffer  = props['buffer']
    ctype   = props.get('type', 'vector')

    # Skip slope constraint — handled in Cell 11 from DEM
    if ctype == 'slope':
        print(f"  ↷ {label:35} derived from DEM in Cell 11, skipped")
        continue

    # Skip if no path defined
    if path is None:
        print(f"  ↷ {label:35} no path defined, skipped")
        continue

    # Skip if file not found in Drive
    if not os.path.exists(path):
        print(f"  ✗ {label:35} file not found in Drive, skipped")
        continue

    clip_vector(path, clipped, label, buffer)

print(f"\nAll clipped layers saved to: {CLIPPED_DIR}")

In [ ]:
# ============================================================
# CELL 7 — AHP Weights (Standard)
# Derives factor weights using the Analytic Hierarchy Process
# (Saaty, 1990). All comparisons read from Cell 2.
# Adding a new group or factor in Cell 2 is automatically
# reflected here — no changes needed in this cell.
# Documents CR values — CR > 0.10 justifies FAHP in Cells 8/9.
# ============================================================


RI = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90,
      5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41,
      9: 1.45, 10: 1.49}

def build_matrix(factors, comparisons):
    n      = len(factors)
    matrix = np.ones((n, n))
    idx    = {f: i for i, f in enumerate(factors)}
    for i, j, score in comparisons:
        matrix[idx[i]][idx[j]] = score
        matrix[idx[j]][idx[i]] = 1 / score
    return matrix

def calculate_weights(matrix):
    col_sums   = matrix.sum(axis=0)
    normalised = matrix / col_sums
    return normalised.mean(axis=1)

def consistency_ratio(matrix, weights):
    n          = len(weights)
    lambda_max = (matrix @ weights / weights).mean()
    CI         = (lambda_max - n) / (n - 1)
    CR         = CI / RI[n] if RI[n] > 0 else 0
    return CR, CI, lambda_max

def factors_from_comparisons(comparisons):
    seen = []
    for i, j, _ in comparisons:
        if i not in seen: seen.append(i)
        if j not in seen: seen.append(j)
    return seen

def print_ahp_results(name, factors, matrix, weights, CR):
    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print("\n  Pairwise Comparison Matrix:")
    header = f"  {'':25}" + "".join(f"{f[:8]:>10}" for f in factors)
    print(header)
    for i, fi in enumerate(factors):
        row = f"  {fi:25}" + "".join(
            f"{matrix[i][j]:>10.3f}" for j in range(len(factors))
        )
        print(row)
    print("\n  Priority Weights:")
    for f, w in zip(factors, weights):
        print(f"    {f:25} {w:.4f}  ({w*100:.1f}%)")
    print(f"\n  Consistency Ratio (CR): {CR:.4f}", end="  ")
    if CR <= 0.10:
        print("✓ Acceptable")
    else:
        print("✗ INCONSISTENT — see Cells 8 and 9 for FAHP solution")
    print()

# ============================================================
# GROUP LEVEL
# ============================================================

group_factors = factors_from_comparisons(AHP_COMPARISONS['group'])
group_matrix  = build_matrix(group_factors, AHP_COMPARISONS['group'])
group_weights_ahp = calculate_weights(group_matrix)
group_CR, _, _    = consistency_ratio(group_matrix, group_weights_ahp)
print_ahp_results('GROUP LEVEL', group_factors, group_matrix, group_weights_ahp, group_CR)
GW_AHP = {f: w for f, w in zip(group_factors, group_weights_ahp)}

# ============================================================
# WITHIN-GROUP WEIGHTS
# Loops over all groups in AHP_COMPARISONS automatically.
# ============================================================

group_factor_weights_ahp = {}

for group_name, comparisons in AHP_COMPARISONS.items():
    if group_name == 'group':
        continue
    factors  = factors_from_comparisons(comparisons)
    matrix   = build_matrix(factors, comparisons)
    weights  = calculate_weights(matrix)
    CR, _, _ = consistency_ratio(matrix, weights)
    print_ahp_results(f'GROUP — {group_name.upper()}', factors, matrix, weights, CR)
    group_factor_weights_ahp[group_name] = {
        f: w for f, w in zip(factors, weights)
    }

# ============================================================
# FINAL GLOBAL WEIGHTS — Standard AHP
# ============================================================

final_weights_ahp = {}

for group_name, factor_weights in group_factor_weights_ahp.items():
    if group_name not in GW_AHP:
        continue
    for factor, w in factor_weights.items():
        final_weights_ahp[factor] = GW_AHP[group_name] * w

total = sum(final_weights_ahp.values())
final_weights_ahp = {f: w/total for f, w in final_weights_ahp.items()}

print("=" * 60)
print("  FINAL GLOBAL WEIGHTS — Standard AHP")
print("=" * 60)
print("\n  Factor                    Global Weight")
print("  " + "-" * 45)
for f, w in final_weights_ahp.items():
    bar = "█" * int(w * 40)
    print(f"    {f:25} {w:.4f}  ({w*100:.1f}%)  {bar}")
print(f"\n  Total: {sum(final_weights_ahp.values()):.4f}")
print("\n  ✓ Weights saved to FINAL_WEIGHTS_AHP")
FINAL_WEIGHTS_AHP = final_weights_ahp

Technical Annex Statement
AHP Consistency Analysis
The initial pairwise comparison matrices were constructed following Saaty (1990), using a 1–9 ratio scale to express the relative importance of criteria. Following standard AHP procedure, the Consistency Ratio (CR) was calculated for each matrix to verify the logical coherence of the judgements. A CR ≤ 0.10 is considered acceptable (Saaty, 1990).
Two matrices produced unacceptable consistency ratios:
Group Level matrix: CR = 0.1195
The inconsistency arises from the following logical contradiction. Community Knowledge (D) was judged to be strongly more important than Infrastructure (B) by a score of 7, and equally strongly more important than Socioeconomic factors (C) by a score of 7. However, Hazards (A) were judged to dominate Community Knowledge by only a moderate score of 3. Since both Hazards and Community Knowledge strongly dominate Infrastructure and Socioeconomic factors by equal margins, mathematical consistency requires that Hazards and Community Knowledge be approximately equal in importance. The score of 3 between them violates this requirement, producing a CR above the acceptable threshold.
Group C — Socioeconomic / Planning matrix: CR = 0.4059
The inconsistency arises from a transitivity violation. Settlements were judged to be very strongly more important than Investment nodes (score 7), and Investment nodes were judged to be very strongly more important than Transport corridors (score 7). By the transitivity principle inherent in AHP, Settlements should therefore dominate Transport corridors by a factor of approximately 49 (7 × 7). However, Settlements were only scored 7 over Transport corridors, creating a severe internal contradiction that produced a CR well above the acceptable threshold (Saaty, 1990; Malczewski, 1999).
Decision to Apply Fuzzy AHP
These inconsistencies are not errors of judgement but rather reflect a known limitation of the standard AHP method. As noted by Mosadeghi et al. (2013), the traditional AHP approach is inadequate for overcoming the uncertainty inherent in the decision-making process, resulting in uncertain value judgements when using a crisp numerical scale. The 1–9 integer scale forces decision-makers to express inherently imprecise preferences as exact values, which frequently produces mathematical inconsistencies even when the underlying judgements are reasonable (Sisman and Aydinoglu, 2020).
To address this limitation, the Fuzzy Analytic Hierarchy Process (FAHP) was adopted in place of standard AHP. FAHP extends the standard method by representing pairwise comparison judgements as triangular fuzzy numbers rather than crisp values, allowing each comparison to be expressed as a range (lower, modal, upper) rather than a single integer. This approach explicitly acknowledges the imprecision of human judgement and has been shown to produce more reliable weight estimates in land suitability analysis (Sisman and Aydinoglu, 2020; Ustaoglu and Aydinoglu, 2020).
The triangular fuzzy scale used follows the conversion proposed by Sisman and Aydinoglu (2020), based on the original FAHP scale:
Saaty ScoreTriangular Fuzzy Number1(1, 1, 1)3(2, 3, 4)5(4, 5, 6)7(6, 7, 8)9(9, 9, 9)
References

Saaty, T.L. (1990). How to make a decision: The Analytic Hierarchy Process. European Journal of Operational Research, 48, 9–26.
Mosadeghi, R., Warnken, J., Tomlinson, R. and Mirfenderesk, H. (2013). Uncertainty analysis in the application of multi-criteria decision-making methods in Australian strategic environmental decisions. Journal of Environmental Planning and Management, 56(8), 1097–1124.
Sisman, S. and Aydinoglu, A.C. (2020). Using GIS-based multi-criteria decision analysis techniques in smart cities. The International Archives of the Photogrammetry, Remote Sensing and Spatial Information Sciences, XLIV-4/W3-2020, 383–389.
Ustaoglu, E. and Aydinoglu, A.C. (2020). Suitability evaluation of urban construction land in Pendik district of Istanbul, Turkey. Land Use Policy, 99, 104783.
Malczewski, J. (1999). GIS and Multi Criteria Decision Analysis. New York: Wiley.

In [19]:
# ============================================================
# CELL 8 — FAHP Weights — Chang Extent Analysis
# First FAHP attempt using Chang (1996) extent analysis.
# All comparisons read from Cell 2 automatically.
# Adding a new group or factor in Cell 2 is automatically
# reflected here — no changes needed in this cell.
# Documents rank reversal issue that justifies switch to
# Buckley method in Cell 9.
# ============================================================


FUZZY_SCALE = {
    1: (1, 1, 1), 2: (1, 2, 3), 3: (2, 3, 4),
    4: (3, 4, 5), 5: (4, 5, 6), 6: (5, 6, 7),
    7: (6, 7, 8), 8: (7, 8, 9), 9: (9, 9, 9),
}

def reciprocal_fuzzy(tfn):
    l, m, u = tfn
    return (1/u, 1/m, 1/l)

def build_fuzzy_matrix(factors, comparisons):
    n      = len(factors)
    idx    = {f: i for i, f in enumerate(factors)}
    matrix = [[(1, 1, 1) for _ in range(n)] for _ in range(n)]
    for i, j, score in comparisons:
        tfn = FUZZY_SCALE[score]
        matrix[idx[i]][idx[j]] = tfn
        matrix[idx[j]][idx[i]] = reciprocal_fuzzy(tfn)
    return matrix

def fuzzy_synthetic_extent(matrix):
    n        = len(matrix)
    row_sums = []
    for i in range(n):
        l = sum(matrix[i][j][0] for j in range(n))
        m = sum(matrix[i][j][1] for j in range(n))
        u = sum(matrix[i][j][2] for j in range(n))
        row_sums.append((l, m, u))
    total_l = sum(r[0] for r in row_sums)
    total_m = sum(r[1] for r in row_sums)
    total_u = sum(r[2] for r in row_sums)
    return [(l/total_u, m/total_m, u/total_l) for l, m, u in row_sums]

def degree_of_possibility(M2, M1):
    l1, m1, u1 = M1
    l2, m2, u2 = M2
    if m2 >= m1:   return 1.0
    elif u1 <= l2: return 0.0
    else:          return (u1 - m2) / ((u1 - m2) + (m1 - l1))

def chang_weights(factors, comparisons):
    matrix  = build_fuzzy_matrix(factors, comparisons)
    extents = fuzzy_synthetic_extent(matrix)
    n       = len(factors)
    weights = []
    for i in range(n):
        min_val = min(
            degree_of_possibility(extents[i], extents[j])
            for j in range(n) if j != i
        )
        weights.append(min_val)
    total   = sum(weights)
    weights = [w/total for w in weights] if total > 0 else [1/n]*n
    return np.array(weights)

def factors_from_comparisons(comparisons):
    seen = []
    for i, j, _ in comparisons:
        if i not in seen: seen.append(i)
        if j not in seen: seen.append(j)
    return seen

def print_chang_results(name, factors, weights):
    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print("\n  Factor Weights (Chang extent analysis):")
    for f, w in zip(factors, weights):
        bar = "█" * int(w * 40)
        print(f"    {f:25} {w:.4f}  ({w*100:.1f}%)  {bar}")
    print(f"\n  Sum: {sum(weights):.4f}\n")

# ============================================================
# GROUP LEVEL
# ============================================================

group_factors       = factors_from_comparisons(AHP_COMPARISONS['group'])
group_weights_chang = chang_weights(group_factors, AHP_COMPARISONS['group'])
print_chang_results('GROUP LEVEL', group_factors, group_weights_chang)
GW_CHANG = {f: w for f, w in zip(group_factors, group_weights_chang)}

# ============================================================
# WITHIN-GROUP WEIGHTS
# Loops over all groups in AHP_COMPARISONS automatically.
# ============================================================

group_factor_weights_chang = {}

for group_name, comparisons in AHP_COMPARISONS.items():
    if group_name == 'group':
        continue
    factors = factors_from_comparisons(comparisons)
    weights = chang_weights(factors, comparisons)
    print_chang_results(f'GROUP — {group_name.upper()}', factors, weights)
    group_factor_weights_chang[group_name] = {
        f: w for f, w in zip(factors, weights)
    }

# ============================================================
# FINAL GLOBAL WEIGHTS — Chang
# ============================================================

final_weights_chang = {}

for group_name, factor_weights in group_factor_weights_chang.items():
    if group_name not in GW_CHANG:
        continue
    for factor, w in factor_weights.items():
        final_weights_chang[factor] = GW_CHANG[group_name] * w

total = sum(final_weights_chang.values())
final_weights_chang = {f: w/total for f, w in final_weights_chang.items()}

print("=" * 60)
print("  FINAL GLOBAL WEIGHTS — FAHP Chang (1996)")
print("=" * 60)
print("\n  Factor                    Global Weight")
print("  " + "-" * 45)
for f, w in final_weights_chang.items():
    bar = "█" * int(w * 40)
    print(f"    {f:25} {w:.4f}  ({w*100:.1f}%)  {bar}")
print(f"\n  Total: {sum(final_weights_chang.values()):.4f}")
print("\n  ✓ Weights saved to FINAL_WEIGHTS_CHANG")
FINAL_WEIGHTS_CHANG = final_weights_chang

  GROUP LEVEL

  Factor Weights (Chang extent analysis):
    Hazards                   0.3890  (38.9%)  ███████████████
    Infrastructure            0.3146  (31.5%)  ████████████
    Socioeconomic             0.2964  (29.6%)  ███████████

  Sum: 1.0000

  GROUP — HAZARDS

  Factor Weights (Chang extent analysis):
    Flood                     0.2295  (23.0%)  █████████
    Landslide                 0.2295  (23.0%)  █████████
    Slope                     0.2498  (25.0%)  █████████
    Community                 0.2911  (29.1%)  ███████████

  Sum: 1.0000

  GROUP — INFRASTRUCTURE

  Factor Weights (Chang extent analysis):
    Primary roads             0.1537  (15.4%)  ██████
    Secondary roads           0.1379  (13.8%)  █████
    Tracks                    0.1434  (14.3%)  █████
    Health                    0.1687  (16.9%)  ██████
    Fire                      0.1124  (11.2%)  ████
    Water                     0.1420  (14.2%)  █████
    Electricity               0.1420  (14.2%)  ████

Limitation of Chang's Extent Analysis and Adoption of Buckley's MethodThe initial FAHP weights were calculated using Chang's (1996) extent analysis method. However, this method has been identified in the literature as producing unreliable priority vectors under certain conditions. Wang et al. (2008) demonstrated that Chang's extent analysis does not correctly estimate fuzzy priorities and can assign zero weights to valid criteria or produce rank reversals — where a factor judged as more important receives a lower weight than a less important one. This was observed in Group C of this study, where Transport corridors received a higher weight than Investment nodes despite being judged as less important.To address this, Buckley's (1985) method was adopted as an alternative FAHP weight derivation approach. Buckley's method calculates weights using the geometric mean of each row in the fuzzy comparison matrix, which produces more stable and reliable priority vectors that are consistent with the original pairwise judgements. Unlike Chang's method, Buckley's approach does not suffer from rank reversals and handles extreme comparison scores more robustly (Zheng et al., 2012).References

Chang, D.Y. (1996). Applications of the extent analysis method on fuzzy AHP. European Journal of Operational Research, 95(3), 649–655.
Buckley, J.J. (1985). Fuzzy hierarchical analysis. Fuzzy Sets and Systems, 17(3), 233–247.
Wang, Y.M., Luo, Y. and Hua, Z. (2008). On the extent analysis method for fuzzy AHP and its applications. European Journal of Operational Research, 186(2), 735–747.
Zheng, G., Zhu, N., Tian, Z., Chen, Y. and Sun, B. (2012). Application of a trapezoidal fuzzy AHP method for work safety evaluation and early warning rating of hot and humid environments. Safety Science, 50(2), 228–239.

In [ ]:
# ============================================================
# CELL 9 — FAHP Weights — Buckley Method
# Final adopted method. All comparisons and factor
# definitions read from Cell 2 automatically.
#
# Adding a new factor or group in Cell 2 is automatically
# reflected here on next run — no changes needed in this cell.
#
# References:
#   Buckley (1985), Fuzzy Sets and Systems, 17(3), 233-247
#   Wang et al. (2008), European Journal of Operational
#   Research, 186(2), 735-747
# ============================================================


FUZZY_SCALE = {
    1: (1, 1, 1), 2: (1, 2, 3), 3: (2, 3, 4),
    4: (3, 4, 5), 5: (4, 5, 6), 6: (5, 6, 7),
    7: (6, 7, 8), 8: (7, 8, 9), 9: (9, 9, 9),
}

def reciprocal_fuzzy(tfn):
    l, m, u = tfn
    return (1/u, 1/m, 1/l)

def build_fuzzy_matrix(factors, comparisons):
    n      = len(factors)
    idx    = {f: i for i, f in enumerate(factors)}
    matrix = [[(1, 1, 1) for _ in range(n)] for _ in range(n)]
    for i, j, score in comparisons:
        tfn = FUZZY_SCALE[score]
        matrix[idx[i]][idx[j]] = tfn
        matrix[idx[j]][idx[i]] = reciprocal_fuzzy(tfn)
    return matrix

def buckley_weights(factors, comparisons):
    matrix    = build_fuzzy_matrix(factors, comparisons)
    n         = len(factors)
    geo_means = []
    for i in range(n):
        l = np.prod([matrix[i][j][0] for j in range(n)]) ** (1/n)
        m = np.prod([matrix[i][j][1] for j in range(n)]) ** (1/n)
        u = np.prod([matrix[i][j][2] for j in range(n)]) ** (1/n)
        geo_means.append((l, m, u))
    sum_l         = sum(g[0] for g in geo_means)
    sum_m         = sum(g[1] for g in geo_means)
    sum_u         = sum(g[2] for g in geo_means)
    fuzzy_weights = [(l/sum_u, m/sum_m, u/sum_l) for l, m, u in geo_means]
    crisp         = np.array([(l+m+u)/3 for l, m, u in fuzzy_weights])
    crisp         = crisp / crisp.sum()
    return crisp, fuzzy_weights

def factors_from_comparisons(comparisons):
    seen = []
    for i, j, _ in comparisons:
        if i not in seen: seen.append(i)
        if j not in seen: seen.append(j)
    return seen

def print_buckley_results(name, factors, weights, fuzzy_weights):
    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print(f"\n  {'Factor':25} {'(l, m, u)':22} {'Weight':>8}  {'%':>6}")
    print("  " + "-" * 65)
    for f, (l, m, u), w in zip(factors, fuzzy_weights, weights):
        bar = "█" * int(w * 40)
        print(f"  {f:25} ({l:.3f}, {m:.3f}, {u:.3f})  {w:>8.4f}  ({w*100:>5.1f}%)  {bar}")
    print(f"\n  Sum: {sum(weights):.4f}\n")

# ============================================================
# GROUP LEVEL
# ============================================================

group_factors     = factors_from_comparisons(AHP_COMPARISONS['group'])
group_weights, group_fuzzy = buckley_weights(group_factors, AHP_COMPARISONS['group'])
print_buckley_results('GROUP LEVEL', group_factors, group_weights, group_fuzzy)
GW = {f: w for f, w in zip(group_factors, group_weights)}

# ============================================================
# WITHIN-GROUP WEIGHTS
# Loops over all groups in AHP_COMPARISONS automatically.
# Adding a new group in Cell 2 appears here automatically.
# ============================================================

group_factor_weights = {}  # stores {group: {factor: weight}}

for group_name, comparisons in AHP_COMPARISONS.items():
    if group_name == 'group':
        continue
    factors          = factors_from_comparisons(comparisons)
    weights, fuzzy   = buckley_weights(factors, comparisons)
    print_buckley_results(f'GROUP — {group_name.upper()}', factors, weights, fuzzy)
    group_factor_weights[group_name] = {
        f: w for f, w in zip(factors, weights)
    }

# ============================================================
# AUTO-DETECT AVAILABLE LAYERS
# Derived from FACTORS in Cell 2 — no hardcoding here.
# ============================================================

CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')

LAYER_AVAILABLE = {
    factor: os.path.exists(os.path.join(CLIPPED_DIR, props['clipped']))
    for factor, props in FACTORS.items()
}

print("=" * 60)
print("  AUTO-DETECTED AVAILABLE LAYERS")
print("=" * 60)
for factor, available in LAYER_AVAILABLE.items():
    status = "✓ available" if available else "✗ pending"
    print(f"    {factor:25} {status}")
print()

# ============================================================
# FINAL GLOBAL WEIGHTS
# Only available layers included — weights renormalised.
# ============================================================

print("=" * 60)
print("  FINAL GLOBAL WEIGHTS — FAHP Buckley (1985)")
print("  (Only available layers included)")
print("=" * 60)

final_weights = {}

for group_name, factor_weights in group_factor_weights.items():
    if group_name not in GW:
        continue
    for factor, w in factor_weights.items():
        if LAYER_AVAILABLE.get(factor, False):
            final_weights[factor] = GW[group_name] * w

# Renormalise
total         = sum(final_weights.values())
final_weights = {f: w/total for f, w in final_weights.items()}

print("\n  Factor                    Global Weight")
print("  " + "-" * 45)
for f, w in final_weights.items():
    bar = "█" * int(w * 40)
    print(f"    {f:25} {w:.4f}  ({w*100:.1f}%)  {bar}")

print(f"\n  Total: {sum(final_weights.values()):.4f}")
print("\n  ✓ Weights saved to FINAL_WEIGHTS_BUCKLEY")

FINAL_WEIGHTS_BUCKLEY = final_weights
ACTIVE_WEIGHTS        = FINAL_WEIGHTS_BUCKLEY
print("\n  Active weights: FINAL_WEIGHTS_BUCKLEY")

Treatment of External Infrastructure Influence in Distance-Based Factors

Distance-based factors (e.g. fire stations, health facilities, and infrastructure networks) were standardised using Euclidean distance rasters derived from vector datasets. Initially, distance calculations were performed using a raster grid clipped to the Area of Interest (AOI), based on the Digital Elevation Model (DEM) template. This approach ensured spatial alignment between factors but introduced a methodological limitation: infrastructure features located outside the AOI were excluded during rasterisation, even when their service range extended into the AOI.

As a result, accessibility was underestimated near the AOI boundary, particularly for sparsely distributed services such as fire stations. The issue arises because Euclidean distance transforms require the full spatial distribution of source features to correctly represent proximity gradients.

To address this, the distance calculation was performed on an expanded temporary grid extending beyond the AOI by the maximum distance threshold defined in the scoring scheme (e.g. 8000 m for emergency services). Distances were then reclassified into suitability scores and cropped back to the AOI template grid to ensure consistency with other MCA factors.

This adjustment ensures that infrastructure located outside the AOI appropriately influences suitability values within the study area, thereby avoiding edge effects and improving the spatial validity of accessibility-based criteria.

In [ ]:
# ============================================================
# CELL 10 — Factor Scoring
# Converts each clipped layer into a standardised 1–5 score
# raster ready for WLC aggregation.
#
# Reads FACTORS from Cell 2 automatically.
# Adding a new factor in Cell 2 is automatically scored
# here — no changes needed in this cell.
#
# Scoring direction: 1 = most suitable, 5 = least suitable
# ============================================================


CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')
RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
os.makedirs(RASTERS_DIR, exist_ok=True)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_raster_template():
    """Returns transform, width, height from clipped DEM."""
    with rasterio.open(os.path.join(CLIPPED_DIR, 'dem.tif')) as src:
        if str(src.crs) != TARGET_CRS:
            transform, width, height = calculate_default_transform(
                src.crs, TARGET_CRS, src.width, src.height, *src.bounds
            )
        else:
            transform = src.transform
            width     = src.width
            height    = src.height
    return transform, width, height

def reproject_to_template(src_path, transform, width, height):
    """Reproject raster to match AOI template grid."""
    with rasterio.open(src_path) as src:
        data = np.zeros((height, width), dtype='float32')
        reproject(
            source         = rasterio.band(src, 1),
            destination    = data,
            src_transform  = src.transform,
            src_crs        = src.crs,
            dst_transform  = transform,
            dst_crs        = TARGET_CRS,
            resampling     = Resampling.nearest
        )
    return data

def reclassify(data, breaks, nodata=-9999):
    """Reclassify raster values into 1-5 scores."""
    result = np.full(data.shape, nodata, dtype='float32')
    mask   = data != nodata
    for max_val, score in breaks:
        result[mask & (data <= max_val) & (result == nodata)] = score
    result[mask & (result == nodata)] = breaks[-1][1]
    return result

def distance_score(gpkg_path, breaks, transform, width, height, nodata=-9999):
    """
    Calculate Euclidean distance from vector features, allowing
    features outside the AOI to influence AOI cells.

    Uses an expanded temporary grid based on the maximum break distance,
    then crops back to the original DEM/AOI template grid.
    """
    import math
    from affine import Affine

    gdf = gpd.read_file(gpkg_path).to_crs(TARGET_CRS)

    # If no features, return all nodata
    if gdf.empty:
        return np.full((height, width), nodata, dtype='float32')

    # Pixel size from template
    pixel_size = abs(transform.a)

    # Largest relevant distance, e.g. 8000 m for Fire
    max_dist = max(b[0] for b in breaks)

    # Number of pixels to expand on each side
    pad = int(math.ceil(max_dist / pixel_size))

    # Expanded grid size
    exp_height = height + 2 * pad
    exp_width  = width + 2 * pad

    # Expanded transform: shift origin outward
    exp_transform = Affine(
        transform.a, transform.b, transform.c - pad * pixel_size,
        transform.d, transform.e, transform.f + pad * pixel_size
    )

    # Rasterize features on expanded grid
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]
    burned = rasterio.features.rasterize(
        shapes,
        out_shape=(exp_height, exp_width),
        transform=exp_transform,
        fill=0,
        dtype='uint8'
    )

    # Distance in pixels, then metres
    dist_pixels = distance_transform_edt(burned == 0)
    dist_metres = dist_pixels * pixel_size

    # Reclassify on expanded grid
    scored_exp = np.full((exp_height, exp_width), nodata, dtype='float32')
    for max_val, score in breaks:
        scored_exp[(dist_metres <= max_val) & (scored_exp == nodata)] = score
    scored_exp[scored_exp == nodata] = breaks[-1][1]

    # Crop back to original DEM/AOI template
    result = scored_exp[pad:pad + height, pad:pad + width]

    return result

def save_scored(data, scored_filename, transform, nodata=-9999):
    """Save scored raster to outputs/rasters/"""
    out_path = os.path.join(RASTERS_DIR, scored_filename)
    with rasterio.open(
        out_path, 'w',
        driver    = 'GTiff',
        height    = data.shape[0],
        width     = data.shape[1],
        count     = 1,
        dtype     = 'float32',
        crs       = TARGET_CRS,
        transform = transform,
        nodata    = nodata
    ) as dst:
        dst.write(data.astype('float32'), 1)

# ============================================================
# GET RASTER TEMPLATE
# ============================================================

transform, width, height = get_raster_template()
print(f"Raster template: {width} x {height} px  CRS: {TARGET_CRS}\n")

# ============================================================
# SCORE ALL FACTORS
# Loops over FACTORS from Cell 2 automatically.
# ============================================================

current_group = None

for factor, props in FACTORS.items():

    # Print group header when group changes
    if props['group'] != current_group:
        current_group = props['group']
        print(f"\n{'=' * 60}")
        print(f"  {current_group.upper()}")
        print(f"{'=' * 60}")

    label   = props['label']
    ftype   = props['type']
    clipped = props['clipped']
    scored  = props['scored']
    scoring = props['scoring']

    clipped_path = os.path.join(CLIPPED_DIR, clipped)
    scored_path  = os.path.join(RASTERS_DIR, scored)

    # Skip if already scored
    if os.path.exists(scored_path):
        print(f"  ↷ {label:35} already scored, skipped")
        continue

    # Skip if clipped file does not exist
    if not os.path.exists(clipped_path):
        print(f"  ✗ {label:35} clipped file not found, skipped")
        continue

    # ── Raster — use values directly (already scored 1-5) ───
    if ftype == 'raster':
        data = reproject_to_template(clipped_path, transform, width, height)
        save_scored(data, scored, transform)
        print(f"  ✓ {label:35} → {scored}")

    # ── Slope — derive from DEM ──────────────────────────────
    elif ftype == 'slope':
        dem_data   = reproject_to_template(clipped_path, transform, width, height)
        pixel_size = abs(transform[0])
        dz_dx      = np.gradient(dem_data, pixel_size, axis=1)
        dz_dy      = np.gradient(dem_data, pixel_size, axis=0)
        slope_pct = np.sqrt(dz_dx**2 + dz_dy**2) * 100
        data       = reclassify(slope_pct, scoring)
        save_scored(data, scored, transform)
        print(f"  ✓ {label:35} → {scored}")

    # ── Vector — Euclidean distance then reclassify ──────────
    elif ftype == 'vector':
        data = distance_score(clipped_path, scoring, transform, width, height)
        save_scored(data, scored, transform)
        print(f"  ✓ {label:35} → {scored}")

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print(f"  SCORING COMPLETE")
print(f"{'=' * 60}")

scored_files = sorted([
    f for f in os.listdir(RASTERS_DIR)
    if f.startswith('scored_')
])
print(f"\n  {len(scored_files)} scored rasters in outputs/rasters/:")
for f in scored_files:
    print(f"    {f}")

In [ ]:
# ============================================================
# CELL 10b — Scored Factor Maps
# Plots each scored factor raster individually grouped by
# AHP group. Reads scored filenames from FACTORS in Cell 2.
# Reads visual style from Cell 2b.
# Only plots rasters already produced by Cell 10.
# ============================================================
aoi_raster_mask = rasterio.features.rasterize(
    [(geom, 1) for geom in aoi.to_crs(TARGET_CRS).geometry],
    out_shape = (height, width),
    transform = transform,
    fill      = 0,
    dtype     = 'uint8'
)

with rasterio.open(os.path.join(RASTERS_DIR, 'constraint_mask.tif')) as src:
    constraint_mask = src.read(1).astype('float32')
RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')

print("=" * 60)
print("  CELL 10b — SCORED FACTOR MAPS")
print("=" * 60)

current_group = None
maps_plotted  = 0

for factor, props in FACTORS.items():

    # Print group header when group changes
    if props['group'] != current_group:
        current_group = props['group']
        print(f"\n{current_group}")

    label       = props['label']
    scored      = props['scored']
    scored_path = os.path.join(RASTERS_DIR, scored)

    # Skip if Cell 10 has not produced this scored raster yet
    if not os.path.exists(scored_path):
        print(f"  ✗ {label:35} not yet scored by Cell 10, skipped")
        continue

    print(f"  ✓ {label:35} plotting...")

    # Read scored raster produced by Cell 10
    with rasterio.open(scored_path) as src:
        if str(src.crs) != TARGET_CRS:
            transform_t, width_t, height_t = calculate_default_transform(
                src.crs, TARGET_CRS, src.width, src.height, *src.bounds
            )
            data = np.zeros((height_t, width_t), dtype='float32')
            reproject(
                source        = rasterio.band(src, 1),
                destination   = data,
                src_transform = src.transform,
                src_crs       = src.crs,
                dst_transform = transform_t,
                dst_crs       = TARGET_CRS,
                resampling    = Resampling.nearest
            )
        else:
            data = src.read(1).astype('float32')

    data[data <= 0] = np.nan

    plot_map(
        data,
        f'{label}\n{LAYER_NAMES["aoi"]}',
        scored.replace('.tif', '.png')
    )
    maps_plotted += 1

print(f"\n{'=' * 60}")
print(f"  Maps plotted: {maps_plotted}")
print(f"  ✓ Saved to outputs/maps/")
print("=" * 60)

  The slope map appears predominantly red (score 5) on the
  suitability map because of a spatial coincidence specific
  to the Mzinyati catchment:

  - Flat areas (score 1, slope 0-5%) coincide with the
    valley floor, which is protected by D'MOSS and river
    buffers. These cells are excluded by constraints and
    rendered white on the map.

  - What remains visible (buildable cells) is predominantly
    mid- to upper-slope terrain, which scores 4-5.

  This is not a coding error. It reflects a genuine tension
  in the catchment: environmentally suitable flat land is
  already protected, leaving steep terrain as the de facto
  development zone. This finding is itself significant for
  planning policy.
""")

In [ ]:
# ============================================================
# CELL 11 — Constraint Mask
# Builds binary raster mask from CONSTRAINTS in Cell 2.
# Visual output handled by plot_constraint_map() from Cell 2b.
# ============================================================

# transform / width / height set by get_raster_template() in Cell 2b
transform, width, height = get_raster_template()
mask = np.ones((height, width), dtype='uint8')

print("=" * 60)
print("  CELL 11 — CONSTRAINT MASK")
print("=" * 60)
print(f"\n  Raster template: {width} x {height} px")
print(f"  Initial buildable cells: {mask.sum():,}\n")
print("  Applying constraints...\n")

for key, props in CONSTRAINTS.items():
    label    = props['label']
    ctype    = props['type']
    clipped  = props['clipped']
    buffer   = props['buffer']
    clipped_path = os.path.join(CLIPPED_DIR, clipped)

    if not os.path.exists(clipped_path):
        print(f"  ✗ {label:35} clipped file not found, skipped")
        continue

    if ctype == 'slope':
        threshold = props['threshold']
        with rasterio.open(clipped_path) as src:
            dem_data = np.zeros((height, width), dtype='float32')
            reproject(
                source        = rasterio.band(src, 1),
                destination   = dem_data,
                src_transform = src.transform,
                src_crs       = src.crs,
                dst_transform = transform,
                dst_crs       = TARGET_CRS,
                resampling    = Resampling.bilinear
            )
        pixel_size  = abs(transform[0])
        dz_dx       = np.gradient(dem_data, pixel_size, axis=1)
        dz_dy       = np.gradient(dem_data, pixel_size, axis=0)
        slope_pct   = np.tan(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))) * 100
        constrained = slope_pct > threshold
        excluded    = int(constrained.sum())
        mask[constrained] = 0
        print(f"  ✓ {label:35} excluded: {excluded:,} cells")

    elif ctype == 'vector':
        gdf = gpd.read_file(clipped_path).to_crs(TARGET_CRS)
        if buffer:
            gdf = gdf.copy()
            gdf['geometry'] = gdf.geometry.buffer(buffer)
        shapes  = [(geom, 1) for geom in gdf.geometry if geom is not None]
        burned  = rasterio.features.rasterize(
            shapes,
            out_shape = (height, width),
            transform = transform,
            fill      = 0,
            dtype     = 'uint8'
        )
        excluded = int((burned == 1).sum())
        mask[burned == 1] = 0
        buf_note = f" (buffer: {buffer}m)" if buffer else ""
        print(f"  ✓ {label:35} excluded: {excluded:,} cells{buf_note}")

# ── Save mask ─────────────────────────────────────────────────
mask_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
with rasterio.open(
    mask_path, 'w',
    driver    = 'GTiff',
    height    = height,
    width     = width,
    count     = 1,
    dtype     = 'uint8',
    crs       = TARGET_CRS,
    transform = transform,
    nodata    = 255
) as dst:
    dst.write(mask, 1)

# ── Summary ───────────────────────────────────────────────────
# Count only cells inside AOI
aoi_raster = rasterio.features.rasterize(
    [(geom, 1) for geom in aoi.to_crs(TARGET_CRS).geometry],
    out_shape = (height, width),
    transform = transform,
    fill      = 0,
    dtype     = 'uint8'
)

total_aoi       = int(aoi_raster.sum())
buildable_cells = int(((mask == 1) & (aoi_raster == 1)).sum())
excluded_cells  = int(((mask == 0) & (aoi_raster == 1)).sum())

print(f"\n{'=' * 60}")
print(f"  CONSTRAINT MASK SUMMARY")
print(f"{'=' * 60}")
print(f"  Total AOI cells:  {total_aoi:,}")
print(f"  Buildable cells:  {buildable_cells:,}  ({buildable_cells/total_aoi*100:.1f}% of AOI)")
print(f"  Excluded cells:   {excluded_cells:,}  ({excluded_cells/total_aoi*100:.1f}% of AOI)")
print(f"\n  ✓ Mask saved → constraint_mask.tif")

# ── Plot ──────────────────────────────────────────────────────
plot_constraint_map(mask, 'constraint_mask.png')

In [ ]:
# CELL 12 — Group Suitability Maps
# Applies FAHP within-group weights to scored rasters.
# Visual output handled by functions from Cell 2b.
# ============================================================

# transform / width / height set by get_raster_template() in Cell 2b
transform, width, height = get_raster_template()

# ── Masks ─────────────────────────────────────────────────────
aoi_raster_mask = rasterio.features.rasterize(
    [(geom, 1) for geom in aoi.to_crs(TARGET_CRS).geometry],
    out_shape = (height, width),
    transform = transform,
    fill      = 0,
    dtype     = 'uint8'
)

with rasterio.open(os.path.join(RASTERS_DIR, 'constraint_mask.tif')) as src:
    constraint_mask = src.read(1).astype('float32')

combined_mask  = np.where(
    (aoi_raster_mask == 1) & (constraint_mask == 1),
    1.0, np.nan
)
# AOI only mask — used for WLC scoring without constraint exclusion
aoi_mask_float = np.where(aoi_raster_mask == 1, 1.0, np.nan)

# ── Load scored raster ────────────────────────────────────────
def load_scored(scored_filename):
    path = os.path.join(RASTERS_DIR, scored_filename)
    with rasterio.open(path) as src:
        data = np.zeros((height, width), dtype='float32')
        reproject(
            source        = rasterio.band(src, 1),
            destination   = data,
            src_transform = src.transform,
            src_crs       = src.crs,
            dst_transform = transform,
            dst_crs       = TARGET_CRS,
            resampling    = Resampling.nearest
        )
    data[data <= 0] = np.nan
    return data

# ── Save suitability raster ───────────────────────────────────
def save_suitability(data, filename):
    path = os.path.join(RASTERS_DIR, filename)
    with rasterio.open(
        path, 'w',
        driver    = 'GTiff',
        height    = height,
        width     = width,
        count     = 1,
        dtype     = 'float32',
        crs       = TARGET_CRS,
        transform = transform,
        nodata    = np.nan
    ) as dst:
        dst.write(data, 1)

# ============================================================
# GROUP SUITABILITY MAPS
# ============================================================

print("=" * 60)
print("  CELL 12 — GROUP SUITABILITY MAPS")
print("=" * 60)

group_maps = {}

for group_name, factors in GROUP_MEMBERSHIP.items():

    available = {
        f: FINAL_WEIGHTS_BUCKLEY[f]
        for f in factors
        if f in FINAL_WEIGHTS_BUCKLEY
        and os.path.exists(os.path.join(RASTERS_DIR, FACTORS[f]['scored']))
    }

    if not available:
        print(f"\n  {group_name}: no layers available — skipped")
        continue

    print(f"\n  {group_name}:")

    total_w = sum(available.values())
    norm_w  = {f: w/total_w for f, w in available.items()}

    group_map = np.zeros((height, width), dtype='float32')
    for factor, weight in norm_w.items():
        data       = load_scored(FACTORS[factor]['scored'])
        group_map += weight * np.nan_to_num(data, nan=0)
        print(f"    {factor:25} weight: {weight:.4f}  ({weight*100:.1f}%)")

    # Apply AOI mask only — constraints handled separately in Cell 14
    aoi_mask_float = np.where(aoi_raster_mask == 1, 1.0, np.nan)
    group_map = group_map * aoi_mask_float

    slug     = group_name.lower().replace(' ', '_').replace('/', '_')
    filename = f"suitability_{slug}.tif"
    save_suitability(group_map, filename)
    group_maps[group_name] = group_map

    plot_map(
        group_map,
        f'{group_name} Suitability Map\n{LAYER_NAMES["aoi"]}',
        filename.replace('.tif', '.png')
    )
    print_distribution_table(group_map, group_name)
    plot_distribution_chart(group_map, group_name, slug)

print("\n" + "=" * 60)
print(f"  Group maps produced: {len(group_maps)}")
print(f"  ✓ Saved to outputs/rasters/ and outputs/maps/")
print("=" * 60)

GROUP_MAPS     = group_maps

In [ ]:
import os
from datetime import datetime

print("=" * 60)
print("CHECKING ACTUAL RASTER FILES IN DRIVE")
print("=" * 60)

rasters_dir = os.path.join(OUTPUT_ROOT, 'rasters')

for scenario in SCENARIOS.keys():
    filename = f"suitability_{scenario}.tif"
    filepath = os.path.join(rasters_dir, filename)

    if os.path.exists(filepath):
        # Get file modification time
        mtime = os.path.getmtime(filepath)
        mtime_str = datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')

        # Get file size
        size = os.path.getsize(filepath) / 1024  # KB

        print(f"\n{scenario}:")
        print(f"  Path: {filepath}")
        print(f"  Last modified: {mtime_str}")
        print(f"  Size: {size:.1f} KB")

        # Quick read of first few values
        with rasterio.open(filepath) as src:
            data = src.read(1)
            valid = data[~np.isnan(data)]
            if len(valid) > 0:
                sample = valid[:10]
                print(f"  Sample values (first 10 valid): {sample}")
                unique_sample = np.unique(sample[:100]) if len(sample) > 0 else []
                print(f"  Unique in first 100: {unique_sample}")
    else:
        print(f"\n{scenario}: File NOT FOUND at {filepath}")

In [ ]:
# ============================================================
# CELL 13 — Scenario Analysis
# Produces one suitability map per scenario and compares them.
# Visual output handled by functions from Cell 2b.
# ============================================================

# ── Masks — defined here so Cell 13 runs independently ───────
with rasterio.open(os.path.join(RASTERS_DIR, 'constraint_mask.tif')) as src:
    constraint_mask = src.read(1).astype('float32')

aoi_raster_mask = rasterio.features.rasterize(
    [(geom, 1) for geom in aoi.to_crs(TARGET_CRS).geometry],
    out_shape = (height, width),
    transform = transform,
    fill      = 0,
    dtype     = 'uint8'
)

combined_mask  = np.where(
    (aoi_raster_mask == 1) & (constraint_mask == 1),
    1.0, np.nan
)
aoi_mask_float = np.where(aoi_raster_mask == 1, 1.0, np.nan)

# ============================================================
# CLASSIFICATION FUNCTION (NEW - ADD THIS)
# ============================================================

def classify_to_1_5(data):
    """Classify continuous raster to discrete 1-5 classes using equal intervals."""
    valid_mask = ~np.isnan(data)
    if not valid_mask.any():
        return data

    min_val = np.nanmin(data)
    max_val = np.nanmax(data)

    # Create 5 equal intervals between min and max
    breaks = np.linspace(min_val, max_val, 6)

    classified = np.full_like(data, np.nan)

    for i in range(5):
        if i == 4:
            # Last class includes upper bound
            mask = valid_mask & (data >= breaks[i]) & (data <= breaks[i+1])
        else:
            mask = valid_mask & (data >= breaks[i]) & (data < breaks[i+1])
        classified[mask] = i + 1

    return classified

# ============================================================
# STEP 1 — PRODUCE SCENARIO MAPS
# ============================================================

print("=" * 60)
print("  CELL 13 — SCENARIO ANALYSIS")
print("=" * 60)

scenario_maps = {}

for scenario_name, scenario in SCENARIOS.items():
    description = scenario['description']
    print(f"\n  Scenario: {description}")
    print("  " + "-" * 40)

    scenario_group_weights = {
        k: v for k, v in scenario.items() if k != 'description'
    }

    available_groups = {
        g: scenario_group_weights[g]
        for g in GROUP_MAPS
        if g in scenario_group_weights
    }

    if not available_groups:
        print(f"  No group maps available — skipped")
        continue

    total_gw  = sum(available_groups.values())
    norm_gw   = {g: w/total_gw for g, w in available_groups.items()}

    final_map = np.zeros((height, width), dtype='float32')
    for group_name, weight in norm_gw.items():
        final_map += weight * np.nan_to_num(GROUP_MAPS[group_name], nan=0)
        print(f"    {group_name:25} weight: {weight:.4f}  ({weight*100:.1f}%)")

    final_map = final_map * aoi_mask_float

    # ============================================================
    # CLASSIFY TO 1-5 (REPLACES the old min_val/max_val lines)
    # ============================================================

    print(f"    Classifying: {np.nanmin(final_map):.3f} to {np.nanmax(final_map):.3f} → 1-5")
    final_map = classify_to_1_5(final_map)

    filename = f"suitability_{scenario_name}.tif"
    save_suitability(final_map, filename)
    scenario_maps[scenario_name] = final_map

    plot_map(
        final_map,
        f'Housing Suitability — {description}\n{LAYER_NAMES["aoi"]}',
        f"suitability_{scenario_name}.png"
    )
    print_distribution_table(final_map, description)
    plot_distribution_chart(final_map, description, scenario_name)

# ============================================================
# STEP 2 — SIDE BY SIDE COMPARISON MAP
# ============================================================

print("\n  Side by side scenario comparison...")
plot_comparison_map(scenario_maps, list(scenario_maps.keys()))

# ============================================================
# STEP 3 — DIFFERENCE MAPS
# ============================================================

print("\n  Difference maps...")

scenario_list = list(scenario_maps.items())
pairs = [
    (scenario_list[i], scenario_list[j])
    for i in range(len(scenario_list))
    for j in range(i+1, len(scenario_list))
]

for (name_a, map_a), (name_b, map_b) in pairs:
    desc_a = SCENARIOS[name_a]['description'].split(' — ')[0]
    desc_b = SCENARIOS[name_b]['description'].split(' — ')[0]
    diff   = map_a - map_b

    plot_difference_map(
        diff,
        f'Difference Map — {desc_a} vs {desc_b}\n{LAYER_NAMES["aoi"]}',
        f"difference_{name_a}_vs_{name_b}.png"
    )

# ============================================================
# STEP 4 — GROUPED DISTRIBUTION COMPARISON CHART
# ============================================================

print("\n  Distribution comparison across scenarios...")

print(f"\n  {'─' * 85}")
print(f"  {'Class':<22}", end="")
for scenario_name in scenario_maps:
    desc = SCENARIOS[scenario_name]['description'].split(' — ')[0]
    print(f"  {desc:>18}", end="")
print()
print(f"  {'─' * 85}")

for score, slabel in zip(SUIT_SCORES, SUIT_LABELS):
    print(f"  {slabel.split(' — ')[1]:<22}", end="")
    for final_map in scenario_maps.values():
        buildable = ~np.isnan(final_map)
        total_b   = buildable.sum()
        count     = int(((final_map >= score-0.5) &
                         (final_map < score+0.5) & buildable).sum())
        pct       = count / total_b * 100 if total_b > 0 else 0
        print(f"  {pct:>17.1f}%", end="")
    print()
print(f"  {'─' * 85}")

plot_scenario_comparison_chart(scenario_maps)

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("  SCENARIO ANALYSIS COMPLETE")
print("=" * 60)
print(f"\n  Scenarios produced:  {len(scenario_maps)}")
print(f"  Difference maps:     {len(pairs)}")
print(f"\n  ✓ All outputs saved to outputs/maps/")

SCENARIO_MAPS = scenario_maps

============================================================
  POLICY BRIEF — PLACEHOLDER
============================================================

  Audience: Traditional Council (local level) and COGTA

  [TO BE WRITTEN — key points to address:]

  1. WHAT WAS DONE
     A GIS-based housing suitability analysis was conducted
     for the Mzinyati Stream Catchment using FAHP-weighted
     multi-criteria decision analysis. Factors included
     hazard risk, infrastructure accessibility and
     socioeconomic proximity.

  2. WHAT WAS FOUND
     Of 22,196 existing buildings in the catchment:
     - 4,627 (20.8%) are located in environmentally
       constrained areas (steep slopes, D'MOSS, river buffers)
     - 171 (0.8%) are in areas of low suitability
       for housing development
     - 4,798 (21.6%) require priority municipal attention

  3. WHAT THIS MEANS FOR GOVERNANCE
     The question of whether households in constraint areas
     should be upgraded in situ or relocated is a decision
     for the competent authority — the Traditional Council
     and eThekwini Municipality — informed by site-specific
     assessment, community participation and human rights
     considerations as set out in NUSP (2015).

     This analysis provides the spatial evidence base to
     support those decisions. It does not prescribe outcomes.

  4. RECOMMENDED NEXT STEPS
     - Commission site-specific hazard assessments for the
       4,627 buildings in constraint areas
     - Prioritise infrastructure delivery for the 171
       buildings in low suitability zones
     - Engage the community in participatory upgrading planning
       in line with NUSP incremental upgrading principles
     - Use the plot allocation tool (Use Case A) to create a
       transparent spatial record of future housing allocations

  [NOTE: This text requires review and approval by a qualified
  urban planner or policy specialist before submission to
  COGTA or the Traditional Council.]


In [20]:
# ============================================================
# CELL 14 — At-Risk Settlement Analysis
# Overlays existing building footprints with the suitability
# map and constraint mask to identify buildings in unsuitable
# or constrained locations.
#
# Reads from Cell 2:
#   BUILDING_FOOTPRINTS_PATH, CONSTRAINTS, SCENARIOS,
#   SLOPE_CONSTRAINT_PCT, LAYER_NAMES, TARGET_CRS
# Visual style from Cell 2b.
#
# Runs all scenarios automatically — no scenario selection.
# Adding a new constraint or scenario in Cell 2 is
# automatically reflected here.
#
# NUSP alignment: supports in situ incremental upgrading by
# identifying priority areas for mitigation rather than
# relocation (NUSP, 2015, Part 3 Chapter 13).
#
# Outputs:
#   1. Unified summary table per scenario
#   2. Scenario comparison table
#   3. Robustness assessment
#   4. Policy brief placeholder
#   5. at_risk_buildings.gpkg — enriched building layer
# ============================================================

RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
MAPS_DIR    = os.path.join(OUTPUT_ROOT, 'maps')
CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')

print("=" * 60)
print("  CELL 14 — AT-RISK SETTLEMENT ANALYSIS")
print("=" * 60)
print(f"\n  NUSP alignment: in situ incremental upgrading priority")
print(f"  Scenarios: all {len(SCENARIOS)} scenarios analysed")

# ============================================================
# LOAD DATA
# ============================================================

if not os.path.exists(BUILDING_FOOTPRINTS_PATH):
    raise FileNotFoundError(
        f"Building footprints not found:\n  {BUILDING_FOOTPRINTS_PATH}\n"
        f"Upload to Google Drive and confirm path in Cell 2."
    )

buildings = gpd.read_file(BUILDING_FOOTPRINTS_PATH).to_crs(TARGET_CRS)
print(f"\n  Building footprints loaded: {len(buildings):,} buildings")
buildings = gpd.clip(buildings, aoi.to_crs(TARGET_CRS))
print(f"  Buildings within AOI:       {len(buildings):,} buildings")
total_buildings = len(buildings)

# ── Load constraint mask ──────────────────────────────────────
with rasterio.open(os.path.join(RASTERS_DIR, 'constraint_mask.tif')) as src:
    constraint_data      = src.read(1)
    constraint_transform = src.transform

# ── Load first available scenario raster for transform ────────
first_scenario = list(SCENARIOS.keys())[0]
with rasterio.open(os.path.join(RASTERS_DIR, f"suitability_{first_scenario}.tif")) as src:
    suit_transform = src.transform
    suit_shape     = (src.height, src.width)

# ── Derive slope raster ───────────────────────────────────────
dem_path = os.path.join(CLIPPED_DIR, 'dem.tif')
with rasterio.open(dem_path) as src:
    dem_data = np.zeros(suit_shape, dtype='float32')
    reproject(
        source        = rasterio.band(src, 1),
        destination   = dem_data,
        src_transform = src.transform,
        src_crs       = src.crs,
        dst_transform = suit_transform,
        dst_crs       = TARGET_CRS,
        resampling    = Resampling.bilinear
    )
pixel_size = abs(suit_transform[0])
dz_dx      = np.gradient(dem_data, pixel_size, axis=1)
dz_dy      = np.gradient(dem_data, pixel_size, axis=0)
slope_pct  = np.tan(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))) * 100

# ============================================================
# HELPER FUNCTION
# ============================================================

def sample_raster_at_geometry(data, transform, geometry):
    centroid = geometry.centroid
    row, col = rasterio.transform.rowcol(transform, centroid.x, centroid.y)
    if 0 <= row < data.shape[0] and 0 <= col < data.shape[1]:
        val = data[row, col]
        return float(val) if val > 0 else np.nan
    return np.nan

# ============================================================
# IDENTIFY CONSTRAINT OVERLAPS
# ============================================================

print("\n  Identifying constraint overlaps...")

buildings['in_constraint']   = False
buildings['constraint_type'] = 'None'

# ── Slope constraint ──────────────────────────────────────────
buildings['slope_at_centroid'] = buildings.geometry.apply(
    lambda geom: sample_raster_at_geometry(slope_pct, suit_transform, geom)
)
slope_mask = buildings['slope_at_centroid'] > SLOPE_CONSTRAINT_PCT
buildings.loc[slope_mask, 'in_constraint']   = True
buildings.loc[slope_mask, 'constraint_type'] = CONSTRAINTS['slope']['label']

# ── Vector constraints — loop from Cell 2 ────────────────────
for key, props in CONSTRAINTS.items():
    if props['type'] != 'vector':
        continue
    clipped_path = os.path.join(CLIPPED_DIR, props['clipped'])
    if not os.path.exists(clipped_path):
        continue
    gdf = gpd.read_file(clipped_path).to_crs(TARGET_CRS)
    if props['buffer']:
        gdf = gdf.copy()
        gdf['geometry'] = gdf.geometry.buffer(props['buffer'])

    buildings_reset = buildings.reset_index(drop=False)
    joined = gpd.sjoin(
        buildings_reset[['index', 'geometry']],
        gdf[['geometry']],
        how='left', predicate='intersects'
    )
    in_constraint_idx = joined[joined['index_right'].notna()]['index'].unique()
    mask = buildings.index.isin(in_constraint_idx)
    buildings.loc[mask & ~buildings['in_constraint'], 'constraint_type'] = props['label']
    buildings.loc[mask, 'in_constraint'] = True

# ============================================================
# SAMPLE SUITABILITY SCORES PER SCENARIO
# ============================================================

print("  Sampling suitability scores for all scenarios...")

scenario_classes = {}
for scenario_name in SCENARIOS:
    suit_path_sc = os.path.join(RASTERS_DIR, f"suitability_{scenario_name}.tif")
    if not os.path.exists(suit_path_sc):
        print(f"  ✗ {scenario_name} not found — run Cell 13 first")
        continue
    with rasterio.open(suit_path_sc) as src:
        data_sc      = src.read(1)
        transform_sc = src.transform

    scores = buildings.geometry.apply(
        lambda geom: sample_raster_at_geometry(data_sc, transform_sc, geom)
    )
    classes = scores.apply(
        lambda s: int(s) if not np.isnan(s) else -1
    )
    scenario_classes[scenario_name] = classes

# ============================================================
# UNIFIED SUMMARY TABLE PER SCENARIO
# ============================================================

print(f"\n{'=' * 60}")
print(f"  AT-RISK SETTLEMENT SUMMARY — ALL SCENARIOS")
print(f"{'=' * 60}")
print(f"  Total buildings in AOI: {total_buildings:,}\n")

for scenario_name, classes in scenario_classes.items():
    description = SCENARIOS[scenario_name]['description']
    b           = buildings.copy()
    b['suitability_class'] = classes

    in_constr    = b[b['in_constraint']]
    scored       = b[~b['in_constraint']]
    at_risk      = scored[scored['suitability_class'] >= 4]
    acceptable   = scored[scored['suitability_class'].isin([1,2,3])]
    edge         = scored[scored['suitability_class'] == -1]
    priority     = len(in_constr) + len(at_risk)

    print(f"\n  Scenario: {description}")
    print(f"  {'─' * 70}")
    print(f"  {'Category':<40} {'Buildings':>10} {'% of total':>12}")
    print(f"  {'─' * 70}")

    print(f"  {'In constraint area':<40} {len(in_constr):>10,} {len(in_constr)/total_buildings*100:>11.1f}%")
    for key, props in CONSTRAINTS.items():
        count = len(b[b['constraint_type'] == props['label']])
        if count > 0:
            print(f"    {'of which: ' + props['label']:<38} {count:>10,} {count/total_buildings*100:>11.1f}%")

    print(f"  {'Score 4-5 (unsuitable)':<40} {len(at_risk):>10,} {len(at_risk)/total_buildings*100:>11.1f}%")
    for score in [4, 5]:
        count = len(scored[scored['suitability_class'] == score])
        if count > 0:
            label = SUIT_LABELS[score-1].split(' — ')[1]
            print(f"    {'of which: score ' + str(score) + ' — ' + label:<38} {count:>10,} {count/total_buildings*100:>11.1f}%")

    print(f"  {'Score 1-3 (acceptable)':<40} {len(acceptable):>10,} {len(acceptable)/total_buildings*100:>11.1f}%")
    for score in [1, 2, 3]:
        count = len(scored[scored['suitability_class'] == score])
        if count > 0:
            label = SUIT_LABELS[score-1].split(' — ')[1]
            print(f"    {'of which: score ' + str(score) + ' — ' + label:<38} {count:>10,} {count/total_buildings*100:>11.1f}%")

    print(f"  {'AOI boundary — no raster data':<40} {len(edge):>10,} {len(edge)/total_buildings*100:>11.1f}%")
    print(f"  {'─' * 70}")

    check = len(in_constr) + len(at_risk) + len(acceptable) + len(edge)
    print(f"  {'Total (check)':<40} {check:>10,} {check/total_buildings*100:>11.1f}%")
    print(f"  {'─' * 70}")
    print(f"  Total priority attention: {priority:,} ({priority/total_buildings*100:.1f}%)")

# ============================================================
# SCENARIO COMPARISON TABLE
# ============================================================

print(f"\n{'=' * 60}")
print(f"  SCENARIO COMPARISON")
print(f"{'=' * 60}\n")

col_w = 20
print(f"  {'Category':<40}", end="")
for n in scenario_classes:
    desc = SCENARIOS[n]['description'].split(' — ')[0]
    print(f"  {desc:>{col_w}}", end="")
print()
print(f"  {'─' * (40 + (col_w + 2) * len(scenario_classes))}")

# ── Categories loop — reads from CONSTRAINTS in Cell 2 ────────
categories = [
    ('In constraint area',
        lambda b: b['in_constraint']),
]
for key, props in CONSTRAINTS.items():
    _label = props['label']
    categories.append((
        '  of which: ' + props['label'],
        lambda b, l=_label: b['constraint_type'] == l
    ))
categories += [
    ('Score 4-5 (unsuitable)',
        lambda b: (~b['in_constraint']) & (b['suitability_class'] >= 4)),
    ('Score 1-3 (acceptable)',
        lambda b: (~b['in_constraint']) & (b['suitability_class'].isin([1,2,3]))),
    ('AOI boundary — no data',
        lambda b: (~b['in_constraint']) & (b['suitability_class'] == -1)),
    ('TOTAL priority attention',
        lambda b: b['in_constraint'] | (b['suitability_class'] >= 4)),
]

for label, condition in categories:
    print(f"  {label:<40}", end="")
    for scenario_name, classes in scenario_classes.items():
        b = buildings.copy()
        b['suitability_class'] = classes
        count = condition(b).sum()
        pct   = count / total_buildings * 100
        print(f"  {str(count) + ' (' + f'{pct:.1f}' + '%)':>{col_w}}", end="")
    print()

print(f"  {'─' * (40 + (col_w + 2) * len(scenario_classes))}")

# ── Robustness assessment ─────────────────────────────────────
priority_counts = []
for scenario_name, classes in scenario_classes.items():
    b = buildings.copy()
    b['suitability_class'] = classes
    count = (b['in_constraint'] | (b['suitability_class'] >= 4)).sum()
    priority_counts.append(count)

spread     = max(priority_counts) - min(priority_counts)
pct_spread = spread / total_buildings * 100
if pct_spread < 2.0:
    verdict = "ROBUST — priority buildings stable across scenarios"
elif pct_spread < 5.0:
    verdict = "MODERATELY SENSITIVE — minor variation across scenarios"
else:
    verdict = "SENSITIVE — priority buildings vary significantly"

print(f"\n  Robustness assessment:")
print(f"  Range across scenarios: {spread:,} buildings ({pct_spread:.1f}%)")
print(f"  Assessment: {verdict}")

# ============================================================
# SAVE ENRICHED BUILDING LAYER
# ============================================================

print(f"\n  Saving enriched building layer...")

# Add all scenario scores to output layer
buildings_out = buildings[['geometry', 'in_constraint', 'constraint_type']].copy()
for scenario_name, classes in scenario_classes.items():
    buildings_out[f'score_{scenario_name}'] = classes

out_dir  = os.path.join(OUTPUT_ROOT, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_gpkg = os.path.join(out_dir, 'at_risk_buildings.gpkg')
buildings_out.to_file(out_gpkg, driver='GPKG')
print(f"  ✓ Saved → at_risk_buildings.gpkg")
print(f"    Attributes: in_constraint, constraint_type, score per scenario")



  CELL 14 — AT-RISK SETTLEMENT ANALYSIS

  NUSP alignment: in situ incremental upgrading priority
  Scenarios: all 3 scenarios analysed

  Building footprints loaded: 269,928 buildings
  Buildings within AOI:       22,196 buildings

  Identifying constraint overlaps...
  Sampling suitability scores for all scenarios...

  AT-RISK SETTLEMENT SUMMARY — ALL SCENARIOS
  Total buildings in AOI: 22,196


  Scenario: Hazard-focused — safety priority
  ──────────────────────────────────────────────────────────────────────
  Category                                  Buildings   % of total
  ──────────────────────────────────────────────────────────────────────
  In constraint area                            4,627        20.8%
    of which: Slope > 30%                       2,670        12.0%
    of which: D'MOSS 2018                         248         1.1%
    of which: Rivers                            1,709         7.7%
  Score 4-5 (unsuitable)                        2,957        13.3%
    o

The majority of at-risk buildings (4,627 of 4,798 in the hazard scenario) are at risk because of environmental constraints — not because of poor suitability scores. This means the infrastructure gap is the secondary problem; the primary problem is people living on steep slopes and in river buffers.

In [21]:
# Final validation - check that rasters are truly classified
print("=" * 60)
print("FINAL VALIDATION: Classified Rasters")
print("=" * 60)

for scenario in SCENARIOS.keys():
    path = os.path.join(RASTERS_DIR, f"suitability_{scenario}.tif")
    with rasterio.open(path) as src:
        data = src.read(1)
        valid = data[~np.isnan(data)]
        unique = np.unique(valid)

    print(f"\n{scenario}:")
    print(f"  Unique values: {unique}")
    print(f"  All values 1-5? {set(unique).issubset({1,2,3,4,5})}")
    print(f"  Min: {np.min(valid)}, Max: {np.max(valid)}")

FINAL VALIDATION: Classified Rasters

hazard_focused:
  Unique values: [1. 2. 3. 4. 5.]
  All values 1-5? True
  Min: 1.0, Max: 5.0

balanced:
  Unique values: [1. 2. 3. 4. 5.]
  All values 1-5? True
  Min: 1.0, Max: 5.0

infrastructure_focused:
  Unique values: [1. 2. 3. 4. 5.]
  All values 1-5? True
  Min: 1.0, Max: 5.0
